In [ ]:
# Setup and load modules
%load_ext autoreload
%autoreload 2

import astroflow as af
import unyt
import yt
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import UnivariateSpline
import numpy as np
from scipy.optimize import curve_fit
from unyt import unyt_array, unyt_quantity
from matplotlib.colors import LogNorm
from matplotlib.cm import ScalarMappable
from pathlib import Path
import json
from datetime import datetime
from scipy.ndimage import gaussian_filter
from Hdecompose.atomic_frac import atomic_frac
import astropy.units as U
import astropy.constants as const
import os
from dataclasses import dataclass
from scipy.interpolate import interp1d

gadgetUnitsCosmo = {
    "UnitLength_in_cm": 3.0868e+24,
    "UnitMass_in_g": 1.989e+43,
    "UnitVelocity_in_cm_per_s": 100000,
}

log = af.get_logger("Dwarfs Kinematics")
af.log.set_log_level("INFO")

def make_phase_filter(name, temp_threshold):
    @yt.particle_filter(f"{name}", requires=["Temperature"], filtered_type="PartType0")
    def temp_below_threshold(pfilter, data):
        filter = data[pfilter.filtered_type, "Temperature"] < temp_threshold
        return filter
    return temp_below_threshold

make_phase_filter("cold_gas", 2e4)

def _density_squared(field, data):
    return data[("PartType0", "Density")]**2

def _HI_density(field, data):
    if ("PartType0", "atomic_frac") not in data.ds.field_info:
        return data[("PartType0", "HI")] * data[("PartType0", "Density")] * (1 - data[("PartType0", "Metallicity")])
    else:
        return data[("PartType0", "atomic_frac")] * data[("PartType0", "Density")]

def _HI_mass(field, data):
    if ("PartType0", "atomic_frac") not in data.ds.field_info:
        return data[("PartType0", "HI")] * data[("PartType0", "Masses")] * (1 - data[("PartType0", "Metallicity")])
    else:
        return data[("PartType0", "atomic_frac")] * data[("PartType0", "Masses")] 

def _Rahmati_HI(field, data):
    """Rahmati et al. 2013 fitting formula for HI fraction based on density, temperature, and redshift."""
    z = float(data.ds.current_redshift)
    fH_init = 0.752 
    proton_mass_cgs = const.m_p.cgs 

    nH = data[("PartType0", "Density")].to("g/cm**3") * U.g / (U.cm ** 3) * (1 - data[("PartType0", "Metallicity")]) * fH_init / (proton_mass_cgs)
    rho = data[("PartType0", "Density")].to("g/cm**3") * U.g / (U.cm ** 3)
    Temp = data[("PartType0", "Temperature")].to("K").v * U.K

    return atomic_frac(redshift=z, nH=nH, T=Temp, rho=rho, Habundance = None, onlyA1=False, local = False)

# Setup hook: runs per snapshot during load
def setup_agora_snapshot(sim, idx):
    ds = sim[idx]
    bar = yt.data_objects.unions.ParticleUnion("baryon", ["PartType0", "PartType4"])
    ds.add_particle_union(bar)
    ds.add_field(("PartType0", "density_squared"), function=_density_squared, units="g**2/cm**6",
                 sampling_type="particle", force_override=True)
    if not ("PartType0", "HI") in ds.field_info:
        ds.add_field(("PartType0", "atomic_frac"), function=_Rahmati_HI, display_name="HI Fraction",
                    units="", sampling_type="particle", force_override=True, take_log=True)

    ds.add_field(("PartType0", "HI_density"), function=_HI_density, display_name="HI Density",
                    units="Msun/pc**3", sampling_type="particle", force_override=True, take_log=True)
    ds.add_field(("PartType0", "HI_mass"), function=_HI_mass, display_name="HI Mass",
                units="Msun", sampling_type="particle", force_override=True, take_log=True)
    ds.add_particle_filter("cold_gas")

def resolve_radius(s, altsim=None): 
    radius = s.v.r1_cgas.to_value() if altsim is None else altsim.v.r1_cgas.to_value() # kpc
    # If radius too small (e.g. no cold gas), fallback to stellar radius
    if radius < 0.1:
        radius = s.v.r1_star.to_value() if altsim is None else altsim.v.r1_star.to_value()
        if radius < 0.1:
            log.warning(f"Radius for kinematic calculations is very small ({radius:.3f} kpc). Check if cold gas or stars are present. Using half star radius as fallback.")
            radius = s.v.r_half_star.to_value() if altsim is None else altsim.v.r_half_star.to_value()
            if radius < 0.1:
                log.warning(f"Radius for kinematic calculations is still very small ({radius:.3f} kpc) after fallback to half star radius. Check if any particles are present. Using 0.1 kpc.")
                radius = 0.1
    return radius

# Core properties for simulations

def calc_core_prop(s, altsim=None):
    s.d.virial_radius_crit(force_recompute=False, center = s.v.center, label = "r_virial", auto_save=False)
    s.d.virial_radius_mean(force_recompute=False, center = s.v.center, label = "r_virial_mean", auto_save=False)
    s.d.virial_radius_BN(force_recompute=False, center = s.v.center, label = "r_virial_BN", auto_save=False)
    s.d.radius_e_dm(force_recompute=False, center = s.v.center, label = "r_half_dm", auto_save=False)
    s.d.mass_200(force_recompute=False, center = s.v.center, label = "m_200_all", auto_save=False, particle="all")
    s.d.mass_200(force_recompute=False, center = s.v.center, label = "m_200_dm", auto_save=False, particle="PartType1")
    
    _ = s.ds.index
    if ("PartType0", "Masses") in s.ds.field_info:
        s.d.radius_e_star(force_recompute=False, center = s.v.center, label = "r_half_star", auto_save=False)
        s.d.radius_e_gas(force_recompute=False, center = s.v.center, label = "r_half_gas", auto_save=False)
        s.d.mass_200(force_recompute=False, center = s.v.center, label = "m_200_star", auto_save=False, particle="PartType4")
        s.d.mass_200(force_recompute=False, center = s.v.center, label = "m_200_gas", auto_save=False, particle="PartType0")
        s.d.mass_200(force_recompute=False, center = s.v.center, label = "m_200_cgas", auto_save=False, particle="cold_gas")

        s.d.percent_mass_radius(force_recompute=False, center = s.v.center, label = "r_star_95", particle = "PartType4", percent = 0.95, auto_save=False)
        s.d.percent_mass_radius(force_recompute=False, center = s.v.center, label = "r_cgas_95", particle = "cold_gas", percent = 0.95, auto_save=False)
        s.d.percent_mass_radius(force_recompute=False, center = s.v.center, label = "r_all_95", particle = "all", percent = 0.95, auto_save=False)

    pcalcaxis = "cold_gas" if ("PartType0", "Masses") in s.ds.field_info else "PartType1"
    ang_radius = s.v.r_half_star.to_value() if altsim is None else altsim.v.r_half_star.to_value()
    
    s.d.faceon(force_recompute=False, radius = 2*ang_radius, use_particle=True, gas=False, particle=pcalcaxis, temp=None, center = s.v.center, label = "faceon", auto_save=False)
    s.d.edgeon(force_recompute=False, radius = 2*ang_radius, use_particle=True, gas=False, particle=pcalcaxis, temp=None, center = s.v.center, label = "edgeon", auto_save=False)

    if ("PartType0", "Masses") in s.ds.field_info:
        s.d.R_den(force_recompute=False, center=s.v.center, particle="cold_gas", radius=100, density_thresh=1e0, bins=40, axis=s.v.faceon, label="r1_cgas", auto_save=False)
        s.d.R_den(force_recompute=False, center=s.v.center, particle="PartType4", radius=100, density_thresh=1e0, bins=40, axis=s.v.faceon, label="r1_star", auto_save=False)
        s.d.R_den(force_recompute=False, center=s.v.center, particle="PartType0", radius=100, density_thresh=1e0, bins=40, axis=s.v.faceon, label="r1_gas", auto_save=False)

        s.d.mass_in_sphere(force_recompute=False, center = s.v.center, radius = s.v.r1_cgas.to_value(), particle="cold_gas", label="m_r1_cgas", auto_save=False)
        s.d.mass_in_sphere(force_recompute=False, center = s.v.center, radius = s.v.r1_cgas.to_value(), particle="PartType4", label="m_r1_star", auto_save=False)
        s.d.mass_in_sphere(force_recompute=False, center = s.v.center, radius = s.v.r1_cgas.to_value(), particle="PartType0", label="m_r1_gas", auto_save=False)
        s.d.mass_in_sphere(force_recompute=False, center = s.v.center, radius = s.v.r1_cgas.to_value(), particle="PartType1", label="m_r1_dm", auto_save=False)

        s.d.mass_in_sphere(force_recompute=False, center = s.v.center, radius = s.v.r1_gas.to_value(), particle="cold_gas", label="m_r1gas_cgas", auto_save=False)
        s.d.mass_in_sphere(force_recompute=False, center = s.v.center, radius = s.v.r1_gas.to_value(), particle="PartType4", label="m_r1gas_star", auto_save=False)
        s.d.mass_in_sphere(force_recompute=False, center = s.v.center, radius = s.v.r1_gas.to_value(), particle="PartType0", label="m_r1gas_gas", auto_save=False)
        s.d.mass_in_sphere(force_recompute=False, center = s.v.center, radius = s.v.r1_gas.to_value(), particle="PartType1", label="m_r1gas_dm", auto_save=False)

        s.d.R_den(force_recompute=False, center=s.v.center, particle="cold_gas", radius=100, density_thresh=1e-1, bins=40, axis=s.v.faceon, label="r01_cgas", auto_save=False)

    ang_radius_2 = s.v.r1_cgas.to_value() if altsim is None else altsim.v.r1_cgas.to_value()

    s.d.faceon(force_recompute=False, radius = ang_radius_2, use_particle=True, gas=False, particle=pcalcaxis, temp=None, center = s.v.center, label = "faceon_r1", auto_save=False)
    s.d.edgeon(force_recompute=False, radius = ang_radius_2, use_particle=True, gas=False, particle=pcalcaxis, temp=None, center = s.v.center, label = "edgeon_r1", auto_save=False)

# Core kinematic for simulations

def calc_kinematic_prop(s, altsim=None):
    pcalc = "cold_gas" if ("PartType0", "Masses") in s.ds.field_info else "PartType1"
    radius = s.v.r1_cgas.to_value() if altsim is None else altsim.v.r1_cgas.to_value() # kpc

    # If radius too small (e.g. no cold gas), fallback to stellar radius
    if radius < 0.1:
        radius = s.v.r1_star.to_value() if altsim is None else altsim.v.r1_star.to_value()
        if radius < 0.1:
            log.warning(f"Radius for kinematic calculations is very small ({radius:.3f} kpc). Check if cold gas or stars are present. Using half star radius as fallback.")
            radius = s.v.r_half_star.to_value() if altsim is None else altsim.v.r_half_star.to_value()
            if radius < 0.1:
                log.warning(f"Radius for kinematic calculations is still very small ({radius:.3f} kpc) after fallback to half star radius. Check if any particles are present. Using 0.1 kpc.")
                radius = 0.1

    s.d.bulk_v(force_recompute=False, center = s.v.center, label = "v_bulk", radius = radius, auto_save=False, bv_kwargs={"use_gas":False,"use_particles":True, "particle_type":pcalc})
    s.d.v_max(force_recompute=False, center = s.v.center, label = "v_max", radius = radius, auto_save=False, particle="all", bins=50)
    s.t.r_fid(s.v.v_max/35)
    s.d.v_fid(force_recompute=False, center = s.v.center, label = "v_fid", radius = s.v.r_fid.to_value(), auto_save=False, particle="all")
    s.d.cuspyness(force_recompute=False, center = s.v.center, label = "alpha", radius = s.v.r_fid.to_value(), auto_save=False, particle="all", bins=50)
    s.t.eta_rot(s.v.v_fid / s.v.v_max)

    _ = s.ds.index
    if ("PartType0", "Masses") in s.ds.field_info:
        s.d.v_fid(force_recompute=False, center = s.v.center, label = "v_fid_bar", radius = s.v.r_fid.to_value(), auto_save=False, particle="baryon")
        s.t.eta_bar((s.v.v_fid_bar / s.v.v_fid)**2)
        s.d.v_phi(force_recompute=False, center = s.v.center, label = "v_phi", radius = s.v.r_fid.to_value(), auto_save=False, bulk_v = s.v.v_bulk, axis = s.v.faceon, particle="cold_gas")
        s.d.v_disp(force_recompute=False, center = s.v.center, label = "disp_phi", radius = s.v.r_fid.to_value(), auto_save=False, bulk_v = s.v.v_bulk, axis = s.v.faceon, particle="cold_gas")
        s.t.v_sigma(s.v.v_phi / s.v.disp_phi)
        s.d.sfr_young_star(force_recompute=False, center = s.v.center, radius = s.v.r_virial.to_value(), max_age = 20, auto_save=False, label = "sfr_young_star")
        s.t.ssfr(s.v.sfr_young_star / s.v.m_200_star)

def adaptive_bin_merging(edges, mid, counts, min_particles=50, local_log = log):
    """Generalized agglomerative binning for any 1D coordinate."""
    mid_adapt, edges_adapt, counts_adapt = [], [edges[0]], []
    curr_cnt, curr_sum = 0, 0.0
    
    for i in range(len(mid)):
        curr_cnt += counts[i]
        curr_sum += mid[i] * counts[i]
        if curr_cnt >= min_particles:
            mid_adapt.append(curr_sum / curr_cnt if curr_cnt > 0 else mid[i])
            edges_adapt.append(edges[i+1])
            counts_adapt.append(curr_cnt)
            curr_cnt, curr_sum = 0, 0.0
    
    # Merge any remaining particles into the last bin
    if curr_cnt > 0:
        if counts_adapt:
            prev_cnt = counts_adapt[-1]
            counts_adapt[-1] = prev_cnt + curr_cnt
            mid_adapt[-1] = (mid_adapt[-1] * prev_cnt + curr_sum) / counts_adapt[-1]
            edges_adapt[-1] = edges[-1]
        else:
            mid_adapt.append(curr_sum / curr_cnt)
            edges_adapt.append(edges[-1])
            counts_adapt.append(curr_cnt)

    # Warn if min particle numbers is not reached 
    if counts_adapt[-1] < min_particles:
        local_log.warning(f"Outermost bin is starved! Out of desired {min_particles}, only {counts_adapt[-1]} were reached")
        local_log.warning(f"Full particle count per bin is: {counts_adapt}")
            
    return np.array(edges_adapt), np.array(mid_adapt), np.array(counts_adapt)

def build_bins(data, ptype, field, nbins, extrema = None, log_x = False, min_particles = 50, unit = None):
    profile = af.data.profile(data, (ptype,field), (ptype,"particle_ones"), data_args=af.settings.DataConfig(n_bins=nbins, bin_extrema=extrema, log = log_x, accumulate=False, weight_field=None, x_unit = unit))
    amount_of_p = profile[(ptype,"particle_ones")].v
    edge, mid, _ = adaptive_bin_merging(profile.x_bins.v, profile.x.v, amount_of_p, min_particles = min_particles)

    return {"set_bins": [edge], "x_data": unyt_array(mid, profile.x.units)}

# Load simulations
g4_paths = [
    (r"C:\Home\Astro\TFG\Data\AGORA\G4\snapshot_336.hdf5", "GADGET4-AGORA-1e10q"),
    (r"C:\Home\Astro\TFG\Data\kazuki\agora_1e10q_lv10\snapshot_275.hdf5", "GADGET4-CDwarf-1e10q"),
    (r"C:\Home\Astro\TFG\Data\kazuki\agora_1e10v_lv10\snapshot_275.hdf5", "GADGET4-CDwarf-1e10v"),
    *[(fr"C:\Home\Astro\TFG\Data\kazuki\halo{idx}\snapshot_275.hdf5", f"GADGET4-CDwarf-halo{idx}") for idx in [185, 196, 211, 224, 234, 256, 273, 291, 307, 316, 324]]
]

g4_dm_paths = [
    (r"C:\Home\Astro\TFG\Data\kazuki\dm_only\agora_1e10q_lv10\snapshot_016.hdf5", "GADGET4-CDwarf-DM-1e10q"),
    (r"C:\Home\Astro\TFG\Data\kazuki\dm_only\agora_1e10v_lv10\snapshot_016.hdf5", "GADGET4-CDwarf-DM-1e10v"),
    *[(fr"C:\Home\Astro\TFG\Data\kazuki\dm_only\halo{idx}\snapshot_016.hdf5", f"GADGET4-CDwarf-DM-halo{idx}") for idx in [185, 196, 211, 224, 234, 256, 273, 291, 307, 316, 324]]
]

g4 = [af.load(path, name, unit_base=gadgetUnitsCosmo, setup_hooks=[setup_agora_snapshot]) for path, name in g4_paths]
g4_dm = [af.load(path, name, unit_base=gadgetUnitsCosmo) for path, name in g4_dm_paths]
g4_dm.insert(0, g4_dm[0])  # Duplicate DM-1e10q since it's the same for AGORA run

# AREPO-T AGORA simulations
esn = [20,20,20,27,27,27,81,81,81,81]
thot = [3,5,10,8,12,16,1,4,12,16]
arepo = []
for i in range(len(esn)):
    name = f"e{esn[i]}t{str(thot[i]).zfill(2)}"
    arepo.append(af.load(fr"C:\Home\Astro\TFG\Data\arepo\{name}\snap_336.hdf5", f"AREPOT-AGORA-{name}", unit_base = gadgetUnitsCosmo, setup_hooks=[setup_agora_snapshot]))
arepo_dm = af.load("C:\\Home\\Astro\\TFG\\Data\\arepo\\dm_only\\snap_336.hdf5", "AREPOT-AGORA-DM", unit_base = gadgetUnitsCosmo)


In [ ]:
# Observational stuff
@dataclass
class Obs_Galaxy:
    name: str
    v_max: float;          e_v_max: float
    v_flat: float;         e_v_flat: float
    w20: float;            e_w20: float
    w50: float;            e_w50: float
    m_star: float;         e_m_star: float
    m_halo: float;        e_m_halo: float
    RHI: float;            
    MHI: float;            
    Fg: float
    r_1_2_star: float;     
    r_disk: float;         
    rad_profile: np.ndarray
    v_obs_profile: np.ndarray; e_v_obs_profile: np.ndarray
    v_gas_profile: np.ndarray
    v_disk_profile: np.ndarray
    v_bulge_profile: np.ndarray
    v_bar_profile: np.ndarray
    v_fid: float;          e_v_fid: float
    quality: int
    eta_rot: float;        e_eta_rot: float
    eta_bar: float        
    inclination: float
    m_bar: float;          e_m_bar: float
    # LITTLE THINGS properties
    v_disp_profile: np.ndarray
    sdens_profile: np.ndarray
    alpha: float;          e_alpha: float
    rad_dens_profile: np.ndarray = None

# ====================================================================
# PARSING & ERROR PROPAGATION ROUTINE
# ====================================================================
def load_sparc_data(table1_path, btfr2019_path, btfr2016_path, massmodels_path):
    
    def get_data_lines(filepath):
        with open(filepath, 'r') as f:
            lines = f.readlines()
        start_idx = next(i for i, line in enumerate(lines) if line.startswith('-------')) + 1
        return [l.strip() for l in lines[start_idx:] if l.strip()]

    def safe_float(val, replace_zero=False):
        try:
            v = float(val)
            return np.nan if replace_zero and v == 0.0 else v
        except ValueError:
            return np.nan

    # 1. Parse Table 1 (SPARC_Lelli2016c.mrt.txt)
    t1_data = {}
    for line in get_data_lines(table1_path):
        parts = line.split()
        if len(parts) >= 18:
            t1_data[parts[0]] = {
                'D': safe_float(parts[2]),
                'e_D': safe_float(parts[3]),
                'inc': safe_float(parts[5]),
                'L36': safe_float(parts[7]),
                'e_L36': safe_float(parts[8]),
                'reff': safe_float(parts[9]),
                'rdisk': safe_float(parts[11]),
                'mhi': safe_float(parts[13]),
                'rhi': safe_float(parts[14], replace_zero=True),
                'vflat': safe_float(parts[15], replace_zero=True),
                'e_vflat': safe_float(parts[16], replace_zero=True),
                'q': int(parts[17]) if parts[17].isdigit() else 3
            }

    # 2. Parse BTFR 2019 (BTFR_Lelli2019.mrt.txt)
    t2_data = {}
    for line in get_data_lines(btfr2019_path):
        parts = line.split()
        if len(parts) >= 17:
            t2_data[parts[0]] = {
                'logMb': safe_float(parts[1], replace_zero=True),
                'e_logMb': safe_float(parts[2], replace_zero=True),
                'vmax': safe_float(parts[11], replace_zero=True),
                'e_vmax': safe_float(parts[12], replace_zero=True),
                'wp20': safe_float(parts[13], replace_zero=True),
                'e_wp20': safe_float(parts[14], replace_zero=True),
                'wm50': safe_float(parts[15], replace_zero=True),
                'e_wm50': safe_float(parts[16], replace_zero=True),
            }

    # 3. Parse BTFR 2016 (BTFR_Lelli2016a.mrt.txt)
    t3_data = {}
    for line in get_data_lines(btfr2016_path):
        parts = line.split()
        if len(parts) >= 9:
            t3_data[parts[0]] = {
                'Dist': safe_float(parts[1], replace_zero=True),
                'e_Dist': safe_float(parts[2], replace_zero=True),
                'Fg': safe_float(parts[8], replace_zero=True)
            }

    # 4. Parse Mass Models (MassModels_Lelli2016c.mrt.txt)
    mm_data = {}
    for line in get_data_lines(massmodels_path):
        parts = line.split()
        if len(parts) >= 8:
            name = parts[0]
            if name not in mm_data:
                mm_data[name] = {'R':[], 'Vobs':[], 'e_Vobs':[], 'Vgas':[], 'Vdisk':[], 'Vbul':[]}
            mm_data[name]['R'].append(safe_float(parts[2]))
            mm_data[name]['Vobs'].append(safe_float(parts[3]))
            mm_data[name]['e_Vobs'].append(safe_float(parts[4]))
            mm_data[name]['Vgas'].append(safe_float(parts[5]))
            mm_data[name]['Vdisk'].append(safe_float(parts[6]))
            mm_data[name]['Vbul'].append(safe_float(parts[7]))
            
    for k in mm_data:
        for key in mm_data[k]:
            mm_data[k][key] = np.array(mm_data[k][key])

    # 4. Synthesize & Propagate Errors
    galaxies =[]
    for name, d1 in t1_data.items():
        d2 = t2_data.get(name, {})
        d3 = t3_data.get(name, {})
        mm = mm_data.get(name, {})
        if not mm: continue
            
        rad, vobs, e_vobs = mm['R'], mm['Vobs'], mm['e_Vobs']
        vgas, vdisk, vbul = mm['Vgas'], mm['Vdisk'], mm['Vbul']

        D = d3.get('Dist', np.nan)
        e_D = d3.get('e_Dist', np.nan)
        #rel_D = e_D / D if D > 0 else np.nan

        # Stellar mass from SPARC luminosity (with error propagation)
        m_star = d1['L36'] * 0.5 * 1e9 
        e_m_star = d1['e_L36'] * 0.5 * 1e9 
        
        # HI and gas mass from SPARC and BTFR 2016 data (with error propagation)
        MHI = d1['mhi'] * 1e9 
        # We assume delta_stellar-to-mass-ratio is 0.11 as in Lelli+2016
        #e_Mg = np.sqrt(e_m_bar**2 - e_m_star**2 - (d1['L36'] * 1e9 * 10**(0.11))**2 - (2*m_bar * rel_D)**2)
        #e_MHI = 0.10 * MHI

        # Basic Masses
        # Baryonic mass from BTFR 2019 (with error propagation)
        logMb = d2.get('logMb', np.nan)
        if not np.isnan(logMb):
            m_bar = 10**logMb 
        else:
            m_bar = m_star + 1.4*MHI
        e_logMb = d2.get('e_logMb', np.nan)
        e_m_bar = m_bar * np.log(10) * e_logMb

        # Radii
        RHI = d1['rhi'];       
        r_1_2 = d1['reff'];
        r_disk = d1['rdisk'];

        if len(rad) > 1:
            f_vobs = interp1d(rad, vobs, bounds_error=False, fill_value="extrapolate")
            f_e_vobs = interp1d(rad, e_vobs, bounds_error=False, fill_value="extrapolate")
            # Rescaling is done by multiplying the squared velocities by the new M/L ratios (0.5 and 0.7)
            vbar_sq = vgas * np.abs(vgas) + 0.5 * vdisk**2 + 0.7 * vbul**2
            vbar = np.sqrt(np.maximum(vbar_sq, 0))
            
            # Propagate errors for v_bar
            #e_vbar_sq = np.sqrt((0.10 * vgas * np.abs(vgas))**2 + 
            #                    (0.253 * 0.5 * vdisk**2)**2 + 
            #                    (0.253 * 0.7 * vbul**2)**2 +
            #                    (vbar_sq * rel_D)**2)
            #e_vbar = e_vbar_sq / (2 * vbar) # Since d(V) = d(V^2) / 2V   
            f_vbar = interp1d(rad, vbar, bounds_error=False, fill_value="extrapolate")
            #f_e_vbar = interp1d(rad, e_vbar, bounds_error=False, fill_value="extrapolate")
        else:
            print(f"Warning: Only one radius point for galaxy {name}, cannot interpolate rotation curve.")
            
        vmax = d2.get('vmax', np.nan)
        e_vmax = d2.get('e_vmax', np.nan)
        if np.isnan(vmax) or vmax == 0.0:
            vmax = np.nanmax(vobs) 
            e_vmax = e_vobs[np.nanargmax(vobs)]

        # Characteristic velocities at rfid kpc
        rfid = vmax / 35.0

        v_fid = float(f_vobs(rfid))        
        e_v_fid = float(f_e_vobs(rfid))
        vbar_fid = float(f_vbar(rfid))
        #e_v_bar_fid = float(f_e_vbar(rfid))

        # Dimensionless ratios (Santos-Santos+20)
        eta_rot = v_fid / vmax if vmax > 0 else np.nan
        e_eta_rot = eta_rot * np.sqrt((e_v_fid/v_fid)**2 + (e_vmax/vmax)**2) if v_fid > 0 and vmax > 0 else np.nan
        eta_bar = vbar_fid / vmax if vmax > 0 else np.nan
        #e_v_bar_fid = vbar_fid * 0.5 * rel_D # Vbar scales with sqrt(D), so approx rel error is 0.5 * rel_D
        #e_eta_bar = eta_bar * np.sqrt((e_v_bar_fid/vbar_fid)**2 + (e_vmax/vmax)**2) if vbar_fid > 0 and vmax > 0 else np.nan
        
        # Inner slope (alpha) computation & error
        #alpha, r_alpha, e_alpha = np.nan, np.nan, np.nan
        #if len(rad) >= 2:
        #    r1, r2, v1, v2 = rad[0], rad[1], vobs[0], vobs[1]
        #    ev1, ev2 = e_vobs[0], e_vobs[1]
        #    if r1 > 0 and v1 > 0 and r2 > 0 and v2 > 0:
        #        alpha = (np.log10(v2) - np.log10(v1)) / (np.log10(r2) - np.log10(r1))
        #        r_alpha = (r1 + r2) / 2.0
        #        e_alpha = (1.0 / (np.abs(np.log10(r2/r1)) * np.log(10))) * np.sqrt((ev1/v1)**2 + (ev2/v2)**2)
                
        # Dynamical Mass at outermost radius Proxy -> M_dyn = R*V^2/G
        #m_halo, e_m_halo = np.nan, np.nan
        #if len(rad) > 0 and vobs[-1] > 0:
        #    m_halo = 2.325e5 * rad[-1] * (vobs[-1]**2)
        #    e_m_halo = m_halo * np.sqrt(rel_D**2 + (2 * e_vobs[-1] / vobs[-1])**2)
            
        galaxies.append(Obs_Galaxy(
            name=name,
            v_max=vmax, e_v_max=e_vmax,
            v_flat=d1.get('vflat', np.nan), e_v_flat=d1.get('e_vflat', np.nan),
            w20=d2.get('wp20', np.nan), e_w20=d2.get('e_wp20', np.nan), w50=d2.get('wm50', np.nan), e_w50=d2.get('e_wm50', np.nan),
            m_star=m_star, e_m_star=e_m_star,
            #m_halo=m_halo, e_m_halo=e_m_halo,
            RHI=RHI,
            MHI=MHI, #e_MHI=e_MHI,
            Fg=d3.get('Fg', np.nan),
            r_1_2_star=r_1_2,
            r_disk=r_disk,
            rad_profile=rad,
            v_obs_profile=vobs, e_v_obs_profile=e_vobs,
            v_gas_profile=vgas, v_disk_profile=vdisk, v_bulge_profile=vbul,
            v_bar_profile=vbar,
            v_fid=v_fid, e_v_fid=e_v_fid,
            quality=d1.get('q', 3),
            eta_rot=eta_rot, e_eta_rot=e_eta_rot,
            eta_bar=eta_bar,
            inclination=d1.get('inc', np.nan),
            m_bar=m_bar, e_m_bar=e_m_bar,
            m_halo=np.nan, e_m_halo=np.nan,
            alpha=np.nan, e_alpha=np.nan,
            v_disp_profile=None, sdens_profile=None
        ))
    return galaxies

def load_little_things_data(lt_path):
    lt_path = Path(lt_path)
    
    def norm(n): return n.strip().lower().replace('_', '').replace(' ', '')
    def safe_float(v):
        try: return float(v.strip())
        except: return np.nan

    # 1. Parse LT 2D (Oh+15) - table2.txt for Alpha, Mstar, Mgas
    lt2d_props = {}
    with open(lt_path / 'table2.txt', 'r') as f:
        for line in f:
            if line.startswith('-') or line.startswith(' ') or line.startswith('#'): continue
            parts =[p.strip() for p in line.split('|')]
            if len(parts) >= 26:
                name = norm(parts[0])
                mstar_sed = safe_float(parts[26])
                mstar_k = safe_float(parts[25])
                lt2d_props[name] = {
                    'alpha': safe_float(parts[18]),
                    'e_alpha': safe_float(parts[19]),
                    'm_gas': safe_float(parts[24]) * 1e7,
                    'm_star': mstar_sed * 1e7 if not np.isnan(mstar_sed) else mstar_k * 1e7,
                    'm_200': 10**safe_float(parts[22])
                }

    # 2. Parse 2D Profiles (rotdmbar.txt)
    lt2d_profs = {}
    with open(lt_path / 'rotdmbar.txt', 'r') as f:
        for line in f:
            if line.startswith('-') or line.startswith(' ') or line.startswith('#'): continue
            parts =[p.strip() for p in line.split('|')]
            if len(parts) >= 7 and parts[1] == 'Data':
                name = norm(parts[0])
                if name not in lt2d_profs: lt2d_profs[name] = {'R':[], 'V':[], 'eV':[]}
                r03, v03 = safe_float(parts[2]), safe_float(parts[3])
                lt2d_profs[name]['R'].append(safe_float(parts[4]) * r03)
                lt2d_profs[name]['V'].append(safe_float(parts[5]) * v03)
                lt2d_profs[name]['eV'].append(safe_float(parts[6]) * v03)

    # 3. Parse 3D BTFR Properties (Iorio+17)
    lt3d_props = {}
    btfr_path = lt_path / '3D' / 'BTFR_data.txt'
    if btfr_path.exists():
        with open(btfr_path, 'r') as f:
            for line in f:
                if line.startswith('#'): continue
                parts = line.split()
                if len(parts) >= 5:
                    lt3d_props[norm(parts[0])] = {
                        'm_bar': safe_float(parts[1]), 'e_m_bar': safe_float(parts[2]),
                        'v_flat': safe_float(parts[3]), 'e_v_flat': safe_float(parts[4])
                    }

    # 4. Parse 3D Profiles (finalrot folder)
    lt3d_profs = {}
    finalrot_dir = lt_path / '3D' / 'finalrot'
    if finalrot_dir.exists():
        for fname in os.listdir(finalrot_dir):
            if not fname.endswith('.txt'): continue
            name = norm(fname.split('_')[0])
            profs = {'R':[], 'Vc':[], 'eVc':[], 'Vd':[], 'Sdens':[]}
            with open(finalrot_dir / fname, 'r') as f:
                for line in f:
                    if line.startswith('#'): continue
                    parts = line.split()
                    if len(parts) >= 11:
                        profs['R'].append(safe_float(parts[1]))
                        profs['Vc'].append(safe_float(parts[6]))
                        profs['eVc'].append(safe_float(parts[7]))
                        profs['Vd'].append(safe_float(parts[8]))
                        profs['Sdens'].append(safe_float(parts[10]))
            for k in profs: profs[k] = np.array(profs[k])
            lt3d_profs[name] = profs

    # 4.5. Parse full density profiles
    lt2d_dens = {}
    with open(lt_path / 'dendmbar.txt', 'r') as f:
        for line in f:
            if line.startswith('-') or line.startswith(' ') or line.startswith('#'): continue
            parts = [p.strip() for p in line.split('|')]
            if len(parts) >= 4 and parts[1] == 'Data':
                name = norm(parts[0])
                if name not in lt2d_dens: 
                    lt2d_dens[name] = {'R_phys':[], 'rho_rel':[]}
                
                # R_phys = 10^(log(R/R03)) * R03
                r_scaled_log = safe_float(parts[2])
                r03 = lt2d_props.get(name, {}).get('r03', 1.0)
                lt2d_dens[name]['R_phys'].append((10**r_scaled_log) * r03)


    # 5. Synthesize Galaxies
    lt_galaxies =[]
    for name in set(list(lt2d_props.keys()) + list(lt3d_props.keys())):
        has_3d = name in lt3d_profs and len(lt3d_profs[name]['R']) > 0
        has_2d = name in lt2d_profs and len(lt2d_profs[name]['R']) > 0
        if not has_3d and not has_2d: continue
        
        p2d = lt2d_props.get(name, {})
        p3d = lt3d_props.get(name, {})
        dens_data = lt2d_dens.get(name, {})
        
        # Priority: 3D over 2D
        rad = lt3d_profs[name]['R'] if has_3d else np.array(lt2d_profs[name]['R'])
        vobs = lt3d_profs[name]['Vc'] if has_3d else np.array(lt2d_profs[name]['V'])
        evobs = lt3d_profs[name]['eVc'] if has_3d else np.array(lt2d_profs[name]['eV'])
        vdisp = lt3d_profs[name]['Vd'] if has_3d else None
        sdens = lt3d_profs[name]['Sdens'] if has_3d else None
        
        m_star = p2d.get('m_star', np.nan)
        m_gas = p2d.get('m_gas', np.nan)
        m_bar = p3d.get('m_bar', m_star + m_gas if not np.isnan(m_star) else np.nan)
        m_halo = p2d.get('m_200', np.nan)
        
        vmax = np.nanmax(vobs) if len(vobs) > 0 else np.nan
        e_vmax = evobs[np.nanargmax(vobs)] if len(vobs) > 0 else np.nan
        
        f_vobs = interp1d(rad, vobs, bounds_error=False, fill_value="extrapolate") if len(rad) > 1 else None
        f_evobs = interp1d(rad, evobs, bounds_error=False, fill_value="extrapolate") if len(rad) > 1 else None
        
        rfid = vmax/35.0
        v_fid = float(f_vobs(rfid)) if f_vobs else np.nan
        e_v_fid = float(f_evobs(rfid)) if f_evobs else np.nan
        
        eta_rot = v_fid / vmax if vmax > 0 else np.nan
        e_eta_rot = eta_rot * np.sqrt((e_v_fid/v_fid)**2 + (e_vmax/vmax)**2) if v_fid > 0 and vmax > 0 else np.nan
        
        lt_galaxies.append(Obs_Galaxy(
            name=name + " (LT)",
            v_max=vmax, e_v_max=e_vmax,
            v_flat=p3d.get('v_flat', np.nan), e_v_flat=p3d.get('e_v_flat', np.nan),
            w20=np.nan, e_w20=np.nan, w50=np.nan, e_w50=np.nan,
            m_star=m_star, e_m_star=np.nan, 
            RHI=np.nan, MHI=m_gas/1.33, Fg=m_gas/m_bar,
            r_1_2_star=np.nan, r_disk=np.nan,
            rad_profile=rad, v_obs_profile=vobs, e_v_obs_profile=evobs,
            v_gas_profile=None, v_disk_profile=None, v_bulge_profile=None, v_bar_profile=None,
            v_fid=v_fid, e_v_fid=e_v_fid,
            quality=1,
            eta_rot=eta_rot, e_eta_rot=e_eta_rot, eta_bar=np.nan,
            alpha=p2d.get('alpha', np.nan), e_alpha=p2d.get('e_alpha', np.nan),
            inclination=np.nan,
            m_bar=m_bar, e_m_bar=p3d.get('e_m_bar', np.nan),
            m_halo=m_halo, e_m_halo=np.nan,
            v_disp_profile=vdisp, sdens_profile=sdens,
            rad_dens_profile=np.array(dens_data.get('R_phys', [])),
        ))
    return lt_galaxies

# Alpha con SPARC es mala idea
# Alpha con LITTLE THINGS si
# DO not include MHI error en SPARC, y hacer error Mstar en SPARC mejor. No error en v_bar
# LITTLE THINGS 3D > LITTLE THINGS 2D > SPARC

# ====================================================================
# DATA EXTRACTION
# ====================================================================
root_path = Path("C:/Home/Astro/TFG/Data/Observations/SPARC")
lt_path = Path("C:/Home/Astro/TFG/Data/Observations/LITTLE_THINGS") # <--- Add LT path

table1_path = root_path / 'SPARC_Lelli2016c.mrt.txt'
btfr2019_path = root_path / 'BTFR_Lelli2019.mrt.txt'
btfr2016_path = root_path / 'BTFR_Lelli2016a.mrt.txt'
massmodels_path = root_path / 'MassModels_Lelli2016c.mrt.txt'

if not all(os.path.exists(p) for p in[table1_path, btfr2019_path, btfr2016_path, massmodels_path, lt_path]):
    print("Error: Could not find one or more required SPARC or LITTLE THINGS directories/files.")
else:
    print("Parsing SPARC datasets...")
    sparc_gals = load_sparc_data(table1_path, btfr2019_path, btfr2016_path, massmodels_path)
    print("Parsing LITTLE THINGS datasets...")
    lt_gals = load_little_things_data(lt_path)

    # Dictionary merge using standardized names: LT overrides SPARC
    master_dict = {g.name.strip().lower().replace('_', '').replace(' ', ''): g for g in sparc_gals}
    lt_dict = {g.name.replace('(LT)', '').strip().lower().replace('_', '').replace(' ', ''): g for g in lt_gals}
    common_gals = set(master_dict.keys()) & set(lt_dict.keys())
    print(f"SPARC galaxies: {len(master_dict)}, LITTLE THINGS galaxies: {len(lt_dict)}")
    print(f"Overlapping galaxies (by standardized name): {len(common_gals)} which are: {common_gals}")
    master_dict.update(lt_dict)
    observed_galaxies = list(master_dict.values())
    print(f"Extracted {len(observed_galaxies)} unique galaxies successfully.")

    sparc_galaxies = [g for g in observed_galaxies if not g.name.endswith("(LT)")]
    lt_galaxies = [g for g in observed_galaxies if g.name.endswith("(LT)")]

    print("Generating diagnostic plots with error bars...")


In [ ]:
def faceon_fn(s, altsim=None):
    sim_to_use = s if altsim is None else altsim
    minrad = sim_to_use.d.min_radius(force_recompute=False, center=sim_to_use.v.center, particle="all", N=1000, tol=10).to("kpc").to_value()
    rad_to_use = sim_to_use.v.r1_cgas.to_value()
    p_to_use = "cold_gas" if ("PartType0", "Masses") in s.ds.field_info else "PartType1"
    if rad_to_use < minrad:
        log.warning(f"In {s.sim.name} (using {sim_to_use.sim.name} for calculation), R1_cgas ({rad_to_use:.3f} kpc) is smaller than convergence radius ({minrad:.3f} kpc). Using R01_cgas instead ({sim_to_use.v.r01_cgas.to_value():.3f} kpc)")
        rad_to_use = sim_to_use.v.r01_cgas.to_value()
    log.info(f"Using radius {rad_to_use:.3f} kpc for faceon calculation in simulation {s.sim.name} (using {sim_to_use.sim.name} for calculation)")
    return s.d.faceon(force_recompute=False, radius = rad_to_use, use_particle=True, gas=False, particle=p_to_use, temp=None, center = s.v.center, auto_save=True)

height_virial = []
height_r01 = []

for i, sim in enumerate(g4):
    log.info(f"-- Processing simulation: {sim.name} --")
    s = sim.snap[0] # snapshot view
    s.d.R_den(force_recompute=False, center=s.v.center, particle="PartType0", radius=100, density_thresh=1e0, bins=40, axis=s.v.faceon, label="r1_cgas", auto_save=True, mass_field="HI_mass")
    s.d.R_den(force_recompute=False, center=s.v.center, particle="PartType0", radius=100, density_thresh=1e-1, bins=40, axis=s.v.faceon, label="r01_cgas", auto_save=True, mass_field="HI_mass")
    s.d.R_den(force_recompute=False, center=s.v.center, particle="PartType4", radius=100, density_thresh=1e0, bins=40, axis=s.v.faceon, label="r1_star", auto_save=True, mass_field="Masses")
    faceon = faceon_fn(s)

    #height_virial.append(s.d.height_e(force_recompute=False, center=s.v.center, particle="PartType0", radius=s.v.r_virial.to_value(), height=100, axis=faceon, auto_save=True, mass_field="HI_mass"))
    height_r01.append(s.d.height_e(force_recompute=False, center=s.v.center, particle="PartType0", label="h_e_HI", radius=s.v.r01_cgas.to_value(), height=100, axis=faceon, auto_save=True, mass_field="HI_mass"))

    #print(f"Simulation: {sim.name}, Height at R_virial: {height_virial[-1]:.3f} kpc, Height at R01_cgas: {height_r01[-1]:.3f} kpc")


print(f"Height at R_virial: {height_virial}")
print(f"Height at R01_cgas: {height_r01}")
#label="he_HI",

In [ ]:
from pathlib import Path
import json
from datetime import datetime

def load_npz_data(load_data_path):
    """Loads and returns metadata, plot_data_list, and ax2_data_list from an .npz file."""
    log.info(f"-- Loading circular velocity data from: {load_data_path} --")
    plot_data_list = []
    ax2_data_list = []
    no_plot_data = []
    
    with np.load(load_data_path, allow_pickle=True) as npz:
        raw = npz['metadata'].tolist()
        if isinstance(raw, bytes): raw = raw.decode()
        metadata = json.loads(raw)
        
        for key in npz.files:
            if key == 'metadata': continue
            item = npz[key].item() 
            if item.get("no_plot", False):
                no_plot_data.append(item)
            elif item.get("is_ax2", False):
                ax2_data_list.append(item)
            else:
                plot_data_list.append(item)
                
    return metadata, plot_data_list, ax2_data_list, no_plot_data

# Plots whatever velocity curve you design
def plot_velocity_curve(sim = None, radius_fn = None, sel_fn = None, circ_fn = None, load_data_path = None, adaptive_bins = False, sim_dm = None, circ_dm_fn = None, nbins = 50, path_prefix=r"C:\Home\Astro\TFG\Figures\P_alpha\Circ", save_fig = True, save_data = False, ylim = [-4,50], legend_kwargs = {"frameon":True,"labelcolor":"mfc","loc":"upper right","fontsize":12,"ncol":3,"framealpha":0.9}, ax_2_fn = None, fig_kwargs = {"figsize" : (9,7)}):
    fig, ax = plt.subplots(**fig_kwargs)
    plot_data_list = []
    ax2_data_list = []

    sim_name = sim.name if sim else Path(load_data_path).stem.replace("Circ_", "")

    if load_data_path:
        metadata, plot_data_list, ax2_data_list, _ = load_npz_data(load_data_path)
    else:
        log.info(f"-- Computing circular velocity profile for simulation: {sim.name} --")
        s = sim.snap[0]
        if sim_dm: s_dm = sim_dm.snap[0]
            
        if not isinstance(sel_fn, (list, tuple)):
            sel_fn =[sel_fn]*(len(circ_fn) + (len(circ_dm_fn) if sim_dm else 0) + (len(ax_2_fn) if ax_2_fn else 0))

        min_rad = sim.get_derived("min_radius", 0, center=s.v.center, particle="all", N=1000, tol=10).to_value() # kpc
        max_rad = radius_fn(s)

        if sim_dm and circ_dm_fn:
            for i, fn in enumerate(circ_dm_fn if isinstance(circ_dm_fn, (list, tuple)) else[circ_dm_fn]):
                sp_dm = sel_fn[i](s_dm, altsim=s) 
                res = fn(s_dm, sp_dm, (min_rad, max_rad), nbins, adaptive_bins)
                plot_data_list.extend(res if isinstance(res, list) else [res])

        for i, fn in enumerate(circ_fn if isinstance(circ_fn, (list, tuple)) else[circ_fn]):
            sp = sel_fn[len(circ_dm_fn if sim_dm else []) + i](s) 
            res = fn(s, sp, (min_rad, max_rad), nbins, adaptive_bins)
            plot_data_list.extend(res if isinstance(res, list) else [res])

        if ax_2_fn:
            for i, fn in enumerate(ax_2_fn if isinstance(ax_2_fn, (list, tuple)) else [ax_2_fn]):
                sp = sel_fn[len(circ_dm_fn if sim_dm else []) + len(circ_fn if isinstance(circ_fn, (list, tuple)) else[circ_fn]) + i](s) 
                res = fn(s, sp, (min_rad, max_rad), nbins, adaptive_bins)
                # Flag as secondary axis data
                res_list = res if isinstance(res, list) else [res]
                for r in res_list: r["is_ax2"] = True
                ax2_data_list.extend(res_list)
                
        if save_data:
            path_prefix = Path(path_prefix)    
            path_prefix.mkdir(parents=True, exist_ok=True)
            pathname = path_prefix / f"Circ_{sim.name}.npz"
            
            # Pack all data into kwargs for np.savez
            save_dict = {}
            for i, item in enumerate(plot_data_list + ax2_data_list):
                save_dict[f"line_{i}"] = item

            metadata = {"sim_name": sim.name, "min_rad": min_rad, "max_rad": max_rad, "sim_dm_name": sim_dm.name if sim_dm else None, "circ_fn_name": [fn.__name__ for fn in circ_fn] if circ_fn else None, "circ_dm_fn_name": [fn.__name__ for fn in circ_dm_fn] if circ_dm_fn else None, "ax_2_fn_name": [fn.__name__ for fn in ax_2_fn] if ax_2_fn else None, "nbins": nbins, "adaptive_bins": adaptive_bins, "datetime": datetime.now().isoformat()}
            np.savez_compressed(pathname, metadata=json.dumps(metadata), **save_dict)
            log.info(f"Saved data to {pathname}")
    
    # Now execute the plotting
    for item in plot_data_list:
        if item.get("no_plot", False): continue
        style = af.settings.StyleConfig(ax=ax, label=item["label"], **item["style"])
        af.plot.render.line(item["x"], item["y"], style_args=style)

    if ax2_data_list:
        ax2 = ax.twinx()
        for item in ax2_data_list:
            style = af.settings.StyleConfig(ax=ax2, label=item["label"], **item["style"])
            af.plot.render.line(item["x"], item["y"], style_args=style)

    ax.set_title(f"{sim_name.split('-')[-1]}")
    ax.set_ylim(ylim)
    ax.set_xlabel("Radius [kpc]")
    ax.set_ylabel("Velocity [km/s]")

    if ax2_data_list:
        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(lines1 + lines2, labels1 + labels2, **legend_kwargs)
    else:
        ax.legend(**legend_kwargs)
    
    if save_fig:
        path_prefix = Path(path_prefix)    
        path_prefix.mkdir(parents=True, exist_ok=True)
        pathname = path_prefix / f"Circ_{sim_name}.png"
        fig.savefig(pathname, dpi=300)

def get_radius(s, altsim = None):
    sim_to_use = s if altsim is None else altsim
    minrad = sim_to_use.d.min_radius(force_recompute=False, center=sim_to_use.v.center, particle="all", N=1000, tol=10).to("kpc").to_value()
    rad_to_use = sim_to_use.v.r1_cgas.to_value()
    if rad_to_use < minrad:
        log.warning(f"In {s.sim.name} (using {sim_to_use.sim.name} for calculation), R1_cgas ({rad_to_use:.3f} kpc) is smaller than convergence radius ({minrad:.3f} kpc). Using R01_cgas instead ({sim_to_use.v.r01_cgas.to_value():.3f} kpc)")
        rad_to_use = sim_to_use.v.r01_cgas.to_value()
    log.info(f"Using radius {rad_to_use:.3f} kpc for faceon calculation in simulation {s.sim.name} (using {sim_to_use.sim.name} for calculation)")
    return rad_to_use

def get_bulk_face(s, altsim=None, radius_fn = get_radius, weight_field = "HI_mass", particle_field = "PartType0"):
    rad_to_use = radius_fn(s, altsim=altsim)
    v_bulk = s.d.bulk_v_weighted(force_recompute=False, center = s.v.center, radius = rad_to_use, auto_save=True, particle=particle_field, weight_field=weight_field)
    faceon = s.d.faceon_weighted(force_recompute=False, radius = rad_to_use, particle=particle_field, weight=weight_field, center = s.v.center, auto_save=True)
    return (v_bulk, faceon)

def gen_sel_sphere_fn(max_rad = 100, set_vbulk = True, set_normal = True, radius_fn = get_radius, weight_field = "HI_mass", particle_field = "PartType0"):
    def sel_sphere(s, altsim=None):
        sp = s.sim[s.idx].sphere(s.v.center, (max_rad, "kpc"))
        v_bulk, normal = get_bulk_face(s, altsim=altsim, radius_fn = radius_fn, weight_field=weight_field, particle_field=particle_field)
        if set_normal:
            sp.set_field_parameter("normal", normal)
        if set_vbulk:
            sp.set_field_parameter("bulk_velocity", v_bulk)
        return sp
    return sel_sphere

def gen_sel_disk_fn(max_rad = 100, set_vbulk = True, z_fn = lambda s: 10, radius_fn = get_radius, weight_field = "HI_mass", particle_field = "PartType0"):
    def sel_disk(s, altsim=None):
        v_bulk, normal = get_bulk_face(s, altsim=altsim, radius_fn = radius_fn, weight_field=weight_field, particle_field=particle_field)
        sp = s.sim[s.idx].disk(s.v.center, normal, (max_rad, "kpc"), (z_fn(s), "kpc"))
        sp.set_field_parameter("normal", normal)
        if set_vbulk:
            sp.set_field_parameter("bulk_velocity", v_bulk)
        return sp
    return sel_disk

def make_true_circ_fn(color = "k", ptype = "all", label = "Circular Velocity", log_x = True, min_particles = 50, force_no_adapt = True, mass_field = "Masses"):
    def true_circ_fn(snap, data, radii, nbins, adapt):
        extra_data = {}
        if adapt and not force_no_adapt:
            extra_data = build_bins(data, ptype, "particle_position_spherical_radius", nbins, radii, log_x, min_particles, unit = "kpc")

        profile = af.plot.data.profile(data, bin_fields=(ptype,"particle_position_spherical_radius"), field=(ptype,mass_field), data_args=af.settings.DataConfig(n_bins=nbins,x_unit="kpc",unit="Msun", bin_extrema=[(radii[0],radii[1])], log = log_x, accumulate=True, **extra_data))
        raw_data = af.analysis.registry.postpro_fn.get("circ_velocity")(profile,(ptype,mass_field))

        return {
            "label": label, "x": profile.x.in_units("kpc").v, "y": raw_data.in_units("km/s").v,
            "style": {"color": color, "linestyle": "-", "linewidth": 2}
        }

    return true_circ_fn

def make_v_fn(color = "#8ca252", ptype = "PartType0", label = "Velocity", coord_field = "particle_position_cylindrical_radius", field = "particle_velocity_cylindrical_theta", disp = False, linestyle="-", log_x = True, min_particles = 100, mass_field = "HI_mass"):
    def v_kin_fn(snap, data, radii, nbins, adapt):
        extra_data = {}
        if adapt:
            extra_data = build_bins(data, ptype, coord_field, nbins, radii, log_x, min_particles, unit = "kpc")

        args = af.settings.DataConfig(n_bins=nbins,x_unit="kpc",unit="km/s", bin_extrema=[(radii[0],radii[1])], log = log_x, accumulate=False, weight_field=(ptype,mass_field), **extra_data)
        profile = af.data.profile(data, (ptype,coord_field), (ptype,field), data_args=args)
        raw_data = profile.standard_deviation[(ptype,field)] if disp else profile[(ptype,field)]

        return {
            "label": label, "x": profile.x.in_units("kpc").v, "y": raw_data.in_units("km/s").v,
            "style": {"color": color, "linestyle": linestyle, "linewidth": 2}
        }
    
    return v_kin_fn

def make_num_p_fn(color = "k", ptype = "PartType0", label = "Particle Count", coord_field = "particle_position_cylindrical_radius", linestyle="--", log_x = True, custom_ax = True, ylim = None, alpha = 0.5, min_particles = 100):
    def num_p(snap, data, radii, nbins, adapt):
        extra_data = {}
        if adapt:
            extra_data = build_bins(data, ptype, coord_field, nbins, radii, log_x, min_particles, unit = "kpc")

        args = af.settings.DataConfig(n_bins=nbins,x_unit="kpc", bin_extrema=[(radii[0],radii[1])], log = log_x, accumulate=False, weight_field=None, **extra_data)
        profile = af.data.profile(data, (ptype,coord_field), (ptype,"particle_ones"), data_args=args)
        amount_of_p = profile[(ptype,"particle_ones")].v

        return {
            "label": label, "x": profile.x.in_units("kpc").v, "y": amount_of_p,
            "style": {"color": color, "linestyle": linestyle, "linewidth": 2, "alpha": alpha, "ylim": ylim, "ylabel": label} if custom_ax else {"color": color, "linestyle": linestyle, "linewidth": 2, "alpha": alpha}
        }

    return num_p

def make_correction_fn(pressure = True, thermal=False, dispersion = True, anisotropy = True, labels = ["W/ Pressure","W/ Non-cte Disp.","W/ Anisotropy"], colors = ["#a252a2","#52a295","#6652a2"], log_x = False, s = None, weight_fit = False, min_particles = 100, see_fit = False, mass_field = "HI_mass", ptype = "PartType0", thermal_label="W/ Thermal", thermal_color="#680505"):
    def phi_correction_fn(snap, data, radii, nbins, adapt):
        extra_data_ptype = {}
        if adapt:
            extra_data_ptype = build_bins(data, ptype, "particle_position_cylindrical_radius", nbins, radii, log_x, min_particles, unit = "kpc")

        # Create sphere selector
        ds = data.ds
        center = snap.v.center
        sphere_sel = ds.sphere(center, (radii[1]*2, "kpc"))

        # Generate rho
        rho_field = ("all","Masses")
        profile_rho  = af.plot.data.profile(sphere_sel, ("all","particle_position_spherical_radius"),rho_field, data_args=af.plot.settings.DataConfig(n_bins=nbins,x_unit="kpc",unit="Msun", bin_extrema=[(radii[0],radii[1]*1.5)], log = True, accumulate=False, weight_field = None))
        rho_sph = af.analysis.registry.postpro_fn.get("spherical_shell")(profile_rho,rho_field).in_units("Msun/kpc**3").v

        # Generate weights in fit        
        args = af.settings.DataConfig(n_bins=nbins,x_unit="kpc", bin_extrema=[(radii[0],radii[1])], log = log_x, accumulate=False, weight_field=None, **extra_data_ptype)
        profile_n = af.data.profile(data, (ptype,"particle_position_cylindrical_radius"), (ptype,"particle_ones"), data_args=args)
        amount_of_p = profile_n[(ptype,"particle_ones")].v
        ptype_w = np.sqrt(np.maximum(amount_of_p - 1,0) / 2)  if weight_fit else None

        # Generate v_phi, disp_phi and disp_r
        args = af.settings.DataConfig(n_bins=nbins,x_unit="kpc",unit="km/s", bin_extrema=[(radii[0],radii[1])], log = log_x, accumulate=False, weight_field=(ptype,mass_field), **extra_data_ptype)
        v_data = []
        for disp, field in zip([False, True, True],["particle_velocity_cylindrical_theta", "particle_velocity_cylindrical_theta", "particle_velocity_cylindrical_radius"]):
            profile = af.data.profile(data, (ptype,"particle_position_cylindrical_radius"), (ptype,field), data_args=args)
            raw_data = profile.standard_deviation[(ptype,field)] if disp else profile[(ptype,field)]
            v_data.append(raw_data.in_units("km/s").v)
            r = profile.x.in_units("kpc").v
        v_phi, disp_phi, disp_r = v_data

        # Fit the splines and compute the corrections 
        rho_spline_fn = af.analysis.registry.postpro_fn.get("log_spline_fn")(profile_rho.x.in_units("kpc").v, rho_sph, s = s)
        alpha_spline_fn = rho_spline_fn.derivative()
        rho_cyl = 10 ** rho_spline_fn(np.log10(r))
        omega_fn = af.analysis.registry.postpro_fn.get("log_spline_derivative_fn")(r,rho_cyl * disp_r**2, weights = ptype_w, s = s, see_fit = see_fit)

        omega = omega_fn(np.log10(r))
        alpha = alpha_spline_fn(np.log10(r))
        gamma = omega - alpha
        beta = 1 - (disp_phi**2 / (disp_r**2 + 1e-12))

        out_data = []

        # Generate thermal support from the saved SPH pressure field
        alpha_th = None
        thermal_term = 0.0
        if thermal:
            rho_field = ("PartType0","Masses")
            profile_rho  = af.plot.data.profile(data, (rho_field[0],"particle_position_cylindrical_radius"),rho_field, data_args=af.plot.settings.DataConfig(n_bins=nbins,x_unit="kpc",unit="Msun", bin_extrema=[(radii[0],radii[1])], log = log_x, accumulate=False, weight_field = None, **extra_data_ptype))
            rho_gas_cil = (af.analysis.registry.postpro_fn.get("circular_surface")(profile_rho,rho_field)/data.height).in_units("Msun/kpc**3") # Calculates disk volumetric density from cylindrical surface density, assuming a constant height equal to the one used in the disk selector

            args_p = af.settings.DataConfig(n_bins=nbins, x_unit="kpc",bin_extrema=[(radii[0], radii[1])], log=log_x,accumulate=False, weight_field=(ptype, mass_field),**extra_data_ptype)
            profile_p = af.data.profile(data,(ptype, "particle_position_cylindrical_radius"),(ptype, "Pressure"),data_args=args_p)
            p_th = profile_p[(ptype, "Pressure")]

            if getattr(p_th.units, "is_dimensionless", True) or str(p_th.units) == "1":
                p_th = ds.arr(p_th.v, "code_pressure")
                p_th = p_th.in_units("Msun/(kpc*s**2)")

            print(p_th)
            print(rho_gas_cil)
            p_th_spline_fn = af.analysis.registry.postpro_fn.get("log_spline_fn")(profile_p.x.in_units("kpc").v, p_th.v, s=s)
            alpha_th = p_th_spline_fn.derivative()(np.log10(r))
            print(alpha_th)
            P_over_rho = (p_th / rho_gas_cil).in_units("km**2/s**2").v
            print(P_over_rho)
            thermal_term = alpha_th * P_over_rho
            print(thermal_term)

        if pressure:
            v_p = np.sqrt(np.maximum(v_phi**2 - alpha*disp_r**2,0))
            out_data.append({"label": labels[0], "x": r, "y": v_p, "style": {"color": colors[0], "linestyle": "-", "linewidth": 2}})
        if thermal:
            v_pt = np.sqrt(np.maximum(v_phi**2 - alpha*disp_r**2 - thermal_term, 0))
            out_data.append({"label": thermal_label, "x": r, "y": v_pt,"style": {"color": thermal_color, "linestyle": "-", "linewidth": 2}})
        if dispersion:
            v_pd = np.sqrt(np.maximum(v_phi**2 + (- alpha - gamma)*disp_r**2 - thermal_term,0))
            out_data.append({"label": labels[1], "x": r, "y": v_pd, "style": {"color": colors[1], "linestyle": "-", "linewidth": 2}})
        if anisotropy:
            v_pda = np.sqrt(np.maximum(v_phi**2 + (- alpha - gamma - beta)*disp_r**2 - thermal_term,0))
            out_data.append({"label": labels[2], "x": r, "y": v_pda, "style": {"color": colors[2], "linestyle": "-", "linewidth": 2}})
        
        out_data.append({"no_plot":True, "alpha": alpha, "gamma": gamma, "beta": beta, "alpha_th": alpha_th, "thermal_term": thermal_term, "r": r})

        return out_data

    return phi_correction_fn

def neo_make_correction_fn(pressure = True, thermal=False, dispersion = True, anisotropy = True, labels = ["W/ Pressure","W/ Non-cte Disp.","W/ Anisotropy"], colors = ["#a252a2","#52a295","#6652a2"], log_x = False, s = None, weight_fit = False, min_particles = 100, see_fit = False, mass_field = "HI_mass", ptype = "PartType0", thermal_label="W/ Thermal", thermal_color="#680505"):
    def phi_correction_fn(snap, data, radii, nbins, adapt):
        extra_data_ptype = {}
        if adapt:
            extra_data_ptype = build_bins(data, ptype, "particle_position_cylindrical_radius", nbins, radii, log_x, min_particles, unit = "kpc")

        # Create sphere selector
        ds = data.ds
        center = snap.v.center

        # Generate rho
        rho_tracer_field = (ptype, mass_field)
        profile_rho_tracer = af.plot.data.profile(data, (ptype, "particle_position_cylindrical_radius"), rho_tracer_field, data_args=af.plot.settings.DataConfig(n_bins=nbins, x_unit="kpc", unit="Msun", bin_extrema=[(radii[0], radii[1])], log=True, accumulate=False, weight_field=None, **extra_data_ptype))
        rho_tracer_cyl = (af.analysis.registry.postpro_fn.get("circular_surface")(profile_rho_tracer, rho_tracer_field) / data.height).in_units("Msun/kpc**3").v

        # Generate weights in fit        
        args = af.settings.DataConfig(n_bins=nbins, x_unit="kpc", bin_extrema=[(radii[0], radii[1])], log=log_x, accumulate=False, weight_field=None, **extra_data_ptype)
        profile_n = af.data.profile(data, (ptype, "particle_position_cylindrical_radius"), (ptype, "particle_ones"), data_args=args)
        amount_of_p = profile_n[(ptype, "particle_ones")].v
        ptype_w = np.sqrt(np.maximum(amount_of_p - 1, 0) / 2) if weight_fit else None

        # Generate v_phi, disp_phi and disp_r
        args = af.settings.DataConfig(n_bins=nbins,x_unit="kpc",unit="km/s", bin_extrema=[(radii[0],radii[1])], log = log_x, accumulate=False, weight_field=(ptype,mass_field), **extra_data_ptype)
        v_data = []
        for disp, field in zip([False, True, True],["particle_velocity_cylindrical_theta", "particle_velocity_cylindrical_theta", "particle_velocity_cylindrical_radius"]):
            profile = af.data.profile(data, (ptype,"particle_position_cylindrical_radius"), (ptype,field), data_args=args)
            raw_data = profile.standard_deviation[(ptype,field)] if disp else profile[(ptype,field)]
            v_data.append(raw_data.in_units("km/s").v)
            r = profile.x.in_units("kpc").v
        v_phi, disp_phi, disp_r = v_data

        # Fit the splines and compute the corrections 
        rho_spline_fn = af.analysis.registry.postpro_fn.get("log_spline_fn")(r, rho_tracer_cyl, s = s)
        alpha_spline_fn = rho_spline_fn.derivative()
        rho_cyl_fit = 10 ** rho_spline_fn(np.log10(r))
        omega_fn = af.analysis.registry.postpro_fn.get("log_spline_derivative_fn")(r, rho_cyl_fit * disp_r**2, weights = ptype_w, s = s, see_fit = see_fit)

        omega = omega_fn(np.log10(r))
        alpha = alpha_spline_fn(np.log10(r))
        gamma = omega - alpha
        beta = 1 - (disp_phi**2 / (disp_r**2 + 1e-12))

        out_data = []

        # Generate thermal support from the saved SPH pressure field
        alpha_th = None
        thermal_term = 0.0
        if thermal:
            rho_fluid_field = ("PartType0","Masses")
            profile_rho_fluid  = af.plot.data.profile(data, (rho_fluid_field[0],"particle_position_cylindrical_radius"),rho_fluid_field, data_args=af.plot.settings.DataConfig(n_bins=nbins,x_unit="kpc",unit="Msun", bin_extrema=[(radii[0],radii[1])], log = log_x, accumulate=False, weight_field = None, **extra_data_ptype))
            rho_gas_cyl = (af.analysis.registry.postpro_fn.get("circular_surface")(profile_rho_fluid, rho_fluid_field) / data.height).in_units("Msun/kpc**3")

            args_p = af.settings.DataConfig(n_bins=nbins, x_unit="kpc",bin_extrema=[(radii[0], radii[1])], log=log_x,accumulate=False, weight_field=(ptype, mass_field),**extra_data_ptype)
            profile_p = af.data.profile(data,(ptype, "particle_position_cylindrical_radius"),(ptype, "Pressure"),data_args=args_p)
            p_th = profile_p[(ptype, "Pressure")]

            if getattr(p_th.units, "is_dimensionless", True) or str(p_th.units) == "1":
                p_th = ds.arr(p_th.v, "code_pressure").in_units("Msun/(kpc*s**2)")

            p_th_spline_fn = af.analysis.registry.postpro_fn.get("log_spline_fn")(profile_p.x.in_units("kpc").v, p_th.v, s=s)
            alpha_th = p_th_spline_fn.derivative()(np.log10(r))
            P_over_rho = (p_th / rho_gas_cyl).in_units("km**2/s**2").v
            thermal_term = alpha_th * P_over_rho

        if pressure:
            v_p = np.sqrt(np.maximum(v_phi**2 - alpha*disp_r**2,0))
            out_data.append({"label": labels[0], "x": r, "y": v_p, "style": {"color": colors[0], "linestyle": "-", "linewidth": 2}})
        if thermal:
            v_pt = np.sqrt(np.maximum(v_phi**2 - alpha*disp_r**2 - thermal_term, 0))
            out_data.append({"label": thermal_label, "x": r, "y": v_pt,"style": {"color": thermal_color, "linestyle": "-", "linewidth": 2}})
        if dispersion:
            v_pd = np.sqrt(np.maximum(v_phi**2 + (- alpha - gamma)*disp_r**2 - thermal_term,0))
            out_data.append({"label": labels[1], "x": r, "y": v_pd, "style": {"color": colors[1], "linestyle": "-", "linewidth": 2}})
        if anisotropy:
            v_pda = np.sqrt(np.maximum(v_phi**2 + (- alpha - gamma - beta)*disp_r**2 - thermal_term,0))
            out_data.append({"label": labels[2], "x": r, "y": v_pda, "style": {"color": colors[2], "linestyle": "-", "linewidth": 2}})
        
        out_data.append({"no_plot":True, "alpha": alpha, "gamma": gamma, "beta": beta, "alpha_th": alpha_th, "thermal_term": thermal_term, "r": r})

        return out_data

    return phi_correction_fn

nbody_circ = make_true_circ_fn(color="gray", ptype="PartType1", label="N-Body")
baryon_circ = make_true_circ_fn(color="#6b6ecf", ptype="baryon", label="Baryons")
HI_circ = make_true_circ_fn(color="#2ADFC3", ptype="PartType0", label="HI", mass_field="HI_mass")
star_circ = make_true_circ_fn(color="#F0E331", ptype="PartType4", label="Stars")
all_circ = make_true_circ_fn(color="#ad494a", ptype="all", label="All")

v_phi_fn = make_v_fn(color="#8ca252", ptype="PartType0", label=r"$V_{\phi}$", coord_field = "particle_position_cylindrical_radius", field="particle_velocity_cylindrical_theta", log_x = False, mass_field="HI_mass")
v_r_fn = make_v_fn(color="#a28352", ptype="PartType0", label=r"$V_{\rho}$", coord_field = "particle_position_cylindrical_radius", field="particle_velocity_cylindrical_radius", log_x = False, mass_field="HI_mass")
v_z_fn = make_v_fn(color="#529ea2", ptype="PartType0", label=r"$V_{z}$", coord_field = "particle_position_cylindrical_radius", field="particle_velocity_cylindrical_z", log_x = False, mass_field="HI_mass")
v_phi_disp_fn = make_v_fn(color="#8ca252", ptype="PartType0", label=r"$\sigma_{\phi}$", coord_field = "particle_position_cylindrical_radius", field="particle_velocity_cylindrical_theta", disp=True, linestyle="--", log_x = False, mass_field="HI_mass")
v_r_disp_fn = make_v_fn(color="#a28352", ptype="PartType0", label=r"$\sigma_{\rho}$", coord_field = "particle_position_cylindrical_radius", field="particle_velocity_cylindrical_radius", disp=True, linestyle="--", log_x = False, mass_field="HI_mass")
v_z_disp_fn = make_v_fn(color="#529ea2", ptype="PartType0", label=r"$\sigma_{z}$", coord_field = "particle_position_cylindrical_radius", disp=True, linestyle="--", field="particle_velocity_cylindrical_z", log_x = False, mass_field="HI_mass")
n_cil_gas = make_num_p_fn()

all_corrections_fn = make_correction_fn(log_x = False, weight_fit = False, s = None, min_particles = 100, see_fit = True)

v_r_sph = make_v_fn(color="#a28352", ptype="PartType0", label=r"$V_{r}$", coord_field = "particle_position_spherical_radius", field="particle_velocity_spherical_radius", disp=False, linestyle="-", log_x = False, mass_field="HI_mass")
disp_r_sph = make_v_fn(color="#a28352", ptype="PartType0", label=r"$\sigma_{r}$", coord_field = "particle_position_spherical_radius", field="particle_velocity_spherical_radius", disp=True, linestyle="--", log_x = False, mass_field="HI_mass")
v_phi_sph = make_v_fn(color="#8ca252", ptype="PartType0", label=r"$V_{\phi}$", coord_field = "particle_position_spherical_radius", field="particle_velocity_spherical_phi", disp=False, linestyle="-", log_x = False, mass_field="HI_mass")
disp_phi_sph = make_v_fn(color="#8ca252", ptype="PartType0", label=r"$\sigma_{\phi}$", coord_field = "particle_position_spherical_radius", field="particle_velocity_spherical_phi", disp=True, linestyle="--", log_x = False, mass_field="HI_mass")
v_theta_sph = make_v_fn(color="#a252a2", ptype="PartType0", label=r"$V_{\theta}$", coord_field = "particle_position_spherical_radius", field="particle_velocity_spherical_theta", disp=False, linestyle="-", log_x = False, mass_field="HI_mass")
disp_theta_sph = make_v_fn(color="#a252a2", ptype="PartType0", label=r"$\sigma_{\theta}$", coord_field = "particle_position_spherical_radius", field="particle_velocity_spherical_theta", disp=True, linestyle="--", log_x = False, mass_field="HI_mass")
n_sph_cgas = make_num_p_fn(coord_field="particle_position_spherical_radius")


In [ ]:
#sphere = gen_sel_sphere_fn(max_rad = 100, set_vbulk = True, set_normal = True)

def r01_rad(s, altsim = None):
    sim_to_use = s if altsim is None else altsim
    return sim_to_use.v.r01_cgas.to_value()

sphere = gen_sel_sphere_fn(set_vbulk = True, set_normal = True, radius_fn = r01_rad)
disk = gen_sel_disk_fn(z_fn = lambda s: s.v.h_e_HI.to_value(), radius_fn = r01_rad, set_vbulk = True)

selectors = [sphere, sphere, sphere] + [disk for _ in range(8)]

for sim in g4:
    plot_velocity_curve(sim, r01_rad, selectors, circ_fn = [baryon_circ, all_circ, HI_circ, v_phi_fn, v_phi_disp_fn, v_r_fn, v_r_disp_fn, v_z_fn, v_z_disp_fn, all_corrections_fn], ax_2_fn = [n_cil_gas], nbins = 100, adaptive_bins = True, path_prefix=r"C:\Home\Astro\TFG\Figures\Circ_models\Circ_cilinder", save_fig = True, save_data = True, ylim = [-10,60], legend_kwargs = {"frameon":True,"labelcolor":"mfc","loc":"upper right","fontsize":13,"ncol":3,"framealpha":0.9})

In [ ]:
# With thermal corrections

all_corrections_fn = neo_make_correction_fn(thermal = True, log_x = False, weight_fit = False, s = None, min_particles = 100, see_fit = True)

def r01_rad(s, altsim = None):
    sim_to_use = s if altsim is None else altsim
    return sim_to_use.v.r01_cgas.to_value()

def r01_half_rad(s, altsim = None):
    sim_to_use = s if altsim is None else altsim
    return sim_to_use.v.r01_cgas.to_value() / 2

sphere = gen_sel_sphere_fn(set_vbulk = True, set_normal = True, radius_fn = r01_half_rad)
disk = gen_sel_disk_fn(z_fn = lambda s: s.v.h_e_HI.to_value(), radius_fn = r01_half_rad, set_vbulk = True)

selectors = [sphere, sphere, sphere] + [disk for _ in range(8)]

for sim in g4:
    plot_velocity_curve(sim, r01_rad, selectors, circ_fn = [baryon_circ, all_circ, HI_circ, v_phi_fn, v_phi_disp_fn, v_r_fn, v_r_disp_fn, v_z_fn, v_z_disp_fn, all_corrections_fn], ax_2_fn = [n_cil_gas], nbins = 100, adaptive_bins = True, path_prefix=r"C:\Home\Astro\TFG\Figures\Circ_models\Circ_thermal_cilinder_half", save_fig = True, save_data = True, ylim = [-10,60], legend_kwargs = {"frameon":True,"labelcolor":"mfc","loc":"upper right","fontsize":13,"ncol":3,"framealpha":0.9})


In [ ]:
for factor in [0.03125, 0.0625, 0.125, 0.25, 0.5, 1, 2, 4, 8, 16]:
    print(f"height_{factor}xH_e")

In [ ]:
# Parameter varying in corrections

def create_curves(height_fn,radius_fn,particle_count,ptype="PartType0",mass_field="HI_mass", ang_mom_ptype="PartType0", ang_mom_weight="HI_mass", name_suffix="test", bin_n = 100):
    thermal = 1 if ptype == "PartType0" else 0
    def r01_rad(s, altsim = None):
        sim_to_use = s if altsim is None else altsim
        return sim_to_use.v.r01_cgas.to_value()
    
    sphere = gen_sel_sphere_fn(set_vbulk = True, set_normal = True, radius_fn = radius_fn, weight_field = ang_mom_weight, particle_field = ang_mom_ptype)
    disk = gen_sel_disk_fn(z_fn = height_fn, radius_fn = radius_fn, set_vbulk = True, weight_field = ang_mom_weight, particle_field = ang_mom_ptype)
    
    curve_corr_fn = neo_make_correction_fn(thermal = thermal, log_x = False, weight_fit = False, s = None, min_particles = particle_count, see_fit = True, mass_field=mass_field)
    curve_v_phi_fn = make_v_fn(color="#8ca252", ptype=ptype, label=r"$V_{\phi}$", coord_field = "particle_position_cylindrical_radius", field="particle_velocity_cylindrical_theta", log_x = False, mass_field=mass_field, min_particles = particle_count)
    curve_v_r_fn = make_v_fn(color="#a28352", ptype=ptype, label=r"$V_{\rho}$", coord_field = "particle_position_cylindrical_radius", field="particle_velocity_cylindrical_radius", log_x = False, mass_field=mass_field, min_particles = particle_count)
    curve_v_z_fn = make_v_fn(color="#529ea2", ptype=ptype, label=r"$V_{z}$", coord_field = "particle_position_cylindrical_radius", field="particle_velocity_cylindrical_z", log_x = False, mass_field=mass_field, min_particles = particle_count)
    curve_v_phi_disp_fn = make_v_fn(color="#8ca252", ptype=ptype, label=r"$\sigma_{\phi}$", coord_field = "particle_position_cylindrical_radius", field="particle_velocity_cylindrical_theta", disp=True, linestyle="--", log_x = False, mass_field=mass_field, min_particles = particle_count)
    curve_v_r_disp_fn = make_v_fn(color="#a28352", ptype=ptype, label=r"$\sigma_{\rho}$", coord_field = "particle_position_cylindrical_radius", field="particle_velocity_cylindrical_radius", disp=True, linestyle="--", log_x = False, mass_field=mass_field, min_particles = particle_count)
    curve_v_z_disp_fn = make_v_fn(color="#529ea2", ptype=ptype, label=r"$\sigma_{z}$", coord_field = "particle_position_cylindrical_radius", disp=True, linestyle="--", field="particle_velocity_cylindrical_z", log_x = False, mass_field=mass_field, min_particles = particle_count)
    curve_n_cil_gas = make_num_p_fn(min_particles = particle_count)

    selectors = [sphere, sphere, sphere] + [disk for _ in range(8)]
    for sim in g4:
        plot_velocity_curve(sim, r01_rad, selectors, circ_fn = [baryon_circ, all_circ, HI_circ, curve_v_phi_fn, curve_v_phi_disp_fn, curve_v_r_fn, curve_v_r_disp_fn, curve_v_z_fn, curve_v_z_disp_fn, curve_corr_fn], ax_2_fn = [curve_n_cil_gas], nbins = bin_n, adaptive_bins = True, path_prefix=r"C:\Home\Astro\TFG\Figures\Circ_models\Circ_thermal_cilinder_auto_"+name_suffix, save_fig = True, save_data = True, ylim = [-10,60], legend_kwargs = {"frameon":True,"labelcolor":"mfc","loc":"upper right","fontsize":13,"ncol":3,"framealpha":0.9})


In [ ]:
# Height dependence

for factor in [0.03125, 0.0625, 0.125, 0.25, 0.5, 1, 2, 4, 8, 16]:
    height_fn = lambda s, altsim = None, factor = factor: s.v.h_e_HI.to_value() * factor
    radius_fn = lambda s, altsim = None: s.v.r01_cgas.to_value()
    create_curves(height_fn, radius_fn, particle_count = 100, name_suffix=f"height_{factor}xH_e")

In [ ]:
# Radius dependence for ang mom

for factor in [0.03125, 0.0625, 0.125, 0.25, 0.5, 1, 2, 4, 8, 16]:
    height_fn = lambda s, altsim = None, factor = factor: s.v.h_e_HI.to_value()
    radius_fn = lambda s, altsim = None, factor = factor: s.v.r01_cgas.to_value() * factor
    create_curves(height_fn, radius_fn, particle_count = 100, name_suffix=f"radius_{factor}xR01")

In [ ]:
# Part count dependence

for num in [1,10,25,50,90,100,110,150,200,400]:
    height_fn = lambda s, altsim = None, factor = factor: s.v.h_e_HI.to_value()
    radius_fn = lambda s, altsim = None, factor = factor: s.v.r01_cgas.to_value()
    create_curves(height_fn, radius_fn, particle_count = num, name_suffix=f"particle_count_{num}")

In [ ]:
# Bin num dependence

for num in [25,50,90,100,110,150,200,400]:
    height_fn = lambda s, altsim = None: s.v.h_e_HI.to_value()
    radius_fn = lambda s, altsim = None: s.v.r01_cgas.to_value()
    create_curves(height_fn, radius_fn, particle_count = 100, bin_n = num, name_suffix=f"bin_n_{num}")

In [ ]:
# All gas, stars and different ang mom

height_fn = lambda s, altsim = None, factor = factor: s.v.h_e_HI.to_value()
radius_fn = lambda s, altsim = None, factor = factor: s.v.r01_cgas.to_value()
create_curves(height_fn, radius_fn, particle_count = 100, bin_n = 100, name_suffix="all_gas", ptype="PartType0", mass_field="Masses", ang_mom_ptype="PartType0", ang_mom_weight="HI_mass") # Gas
create_curves(height_fn, radius_fn, particle_count = 100, bin_n = 100, name_suffix="all_stars", ptype="PartType4", mass_field="Masses", ang_mom_ptype="PartType0", ang_mom_weight="HI_mass") # Stars
create_curves(height_fn, radius_fn, particle_count = 100, bin_n = 100, name_suffix="ang_mom_gas", ptype="PartType0", mass_field="HI_mass", ang_mom_ptype="PartType0", ang_mom_weight="Masses") # Ang mom with all gas
create_curves(height_fn, radius_fn, particle_count = 100, bin_n = 100, name_suffix="ang_mom_stars", ptype="PartType0", mass_field="HI_mass", ang_mom_ptype="PartType4", ang_mom_weight="Masses") # Ang mom with all stars

In [ ]:
s = g4[5].snap[0]
print(s.d.faceon_weighted(force_recompute=False, radius = s.v.r01_cgas.to_value()/2, particle="PartType0", weight="HI_mass", center = s.v.center, auto_save=True), s.d.faceon_weighted(force_recompute=False, radius = s.v.r01_cgas.to_value(), particle="PartType0", weight="HI_mass", center = s.v.center, auto_save=True))

In [ ]:
# Test

def new_bulk_face(s, altsim=None, radius_fn = lambda s, altsim=None: s.v.r01_cgas.to_value()/2):
    rad_to_use = radius_fn(s, altsim=altsim)
    p_to_use = "PartType0" if ("PartType0", "Masses") in s.ds.field_info else "PartType1"
    v_bulk = s.d.bulk_v_weighted(force_recompute=False, center = s.v.center, radius = rad_to_use, auto_save=True, particle=p_to_use, weight_field="HI_mass")
    faceon = s.d.faceon_weighted(force_recompute=False, radius = rad_to_use, particle=p_to_use, weight="HI_mass", center = s.v.center, auto_save=True)
    return (v_bulk, faceon)

def new_disk_fn(max_rad = 100, set_vbulk = True, z_fn = lambda s: 10, radius_fn = get_radius):
    def sel_disk(s, altsim=None):
        v_bulk, normal = new_bulk_face(s, altsim=altsim)
        sp = s.sim[s.idx].disk(s.v.center, normal, (max_rad, "kpc"), (z_fn(s), "kpc"))
        sp.set_field_parameter("normal", normal)
        if set_vbulk:
            sp.set_field_parameter("bulk_velocity", v_bulk)
        return sp
    return sel_disk

all_corrections_fn = neo_make_correction_fn(thermal = True, log_x = False, weight_fit = False, s = None, min_particles = 100, see_fit = True)

def r01_rad(s, altsim = None):
    sim_to_use = s if altsim is None else altsim
    return sim_to_use.v.r01_cgas.to_value()

sphere = gen_sel_sphere_fn(set_vbulk = True, set_normal = True, radius_fn = r01_rad)
disk = new_disk_fn(z_fn = lambda s: s.v.h_e_HI.to_value(), radius_fn = r01_rad, set_vbulk = True)

selectors = [sphere, sphere, sphere] + [disk for _ in range(8)]

for sim in [g4[5]]:
    plot_velocity_curve(sim, r01_rad, selectors, circ_fn = [baryon_circ, all_circ, HI_circ, v_phi_fn, v_phi_disp_fn, v_r_fn, v_r_disp_fn, v_z_fn, v_z_disp_fn, all_corrections_fn], ax_2_fn = [n_cil_gas], nbins = 100, adaptive_bins = True, path_prefix=r"C:\Home\Astro\TFG\Figures\Circ_models\Test", save_fig = True, save_data = False, ylim = [-10,60], legend_kwargs = {"frameon":True,"labelcolor":"mfc","loc":"upper right","fontsize":13,"ncol":3,"framealpha":0.9})

In [ ]:
def interp_profile(x_src, y_src, x_tgt):
    """
    Safely interpolate y(x) onto x_tgt.
    Sorts x_src, removes non-finite values, and returns NaN outside the range.
    """
    x_src = np.asarray(x_src, dtype=float)
    y_src = np.asarray(y_src, dtype=float)
    x_tgt = np.asarray(x_tgt, dtype=float)

    m = np.isfinite(x_src) & np.isfinite(y_src)
    x_src = x_src[m]
    y_src = y_src[m]

    order = np.argsort(x_src)
    x_src = x_src[order]
    y_src = y_src[order]

    if len(x_src) < 2:
        return np.full_like(x_tgt, np.nan, dtype=float)

    return np.interp(x_tgt, x_src, y_src, left=np.nan, right=np.nan)

def get_item_by_label(data, substring, ykey="y"):
    """
    Returns (x, y, item) for the first dict whose label contains substring.
    """
    item = next(item for item in data if substring in item.get("label", ""))
    x = np.asarray(item["x"], dtype=float)
    y = np.asarray(item[ykey], dtype=float)
    return x, y, item

# --- Data Extraction Loop ---
vphi_list = []
vcorr_p_list = []
vcorr_t_list = []
vcorr_d_list = []
vcorr_a_list = []

vphi_list_r1 = []
vcorr_p_list_r1 = []
vcorr_t_list_r1 = []
vcorr_d_list_r1 = []
vcorr_a_list_r1 = []

vmax_phi_list = []
vfid_phi_list = []
vmax_corr_p_list = []
vfid_corr_p_list = []
vmax_corr_t_list = []
vfid_corr_t_list = []
vmax_corr_d_list = []
vfid_corr_d_list = []
vmax_corr_a_list = []
vfid_corr_a_list = []

full_v_data = []
for i, sim in enumerate(g4):
    s = sim.snap[0]

    meta_v, plot_data_v, _, no_plot_data = load_npz_data(
        rf"C:\Home\Astro\TFG\Figures\Circ_models\Circ_thermal_cilinder\Circ_{sim.name}.npz"
    )

    print(f"Extracting data for {meta_v['sim_name']}...")

    # --- Extract profiles on their native grids ---
    r_gas, v_phi, _ = get_item_by_label(plot_data_v, r"$V_{\phi}$")
    _, sigma_r, _ = get_item_by_label(plot_data_v, r"$\sigma_{\rho}$")
    _, sigma_phi, _ = get_item_by_label(plot_data_v, r"$\sigma_{\phi}$")

    r_vcirc, v_circ_raw, _ = get_item_by_label(plot_data_v, "All")
    r_vcorr, v_corr_p_raw, _ = get_item_by_label(plot_data_v, "W/ Pressure")
    _, v_corr_t_raw, _ = get_item_by_label(plot_data_v, "W/ Thermal")
    _, v_corr_d_raw, _ = get_item_by_label(plot_data_v, "W/ Non-cte Disp.")
    _, v_corr_a_raw, _ = get_item_by_label(plot_data_v, "W/ Anisotropy")

    # Beta usually lives in the hidden data; keep it flexible
    beta_item = no_plot_data[0]
    beta_raw = np.asarray(beta_item["beta"], float)
    alpha_th = np.asarray(beta_item.get("alpha_th"), float)
    thermal_term = np.asarray(beta_item.get("thermal_term"), float)

    beta = np.interp(r_gas, r_vcorr, beta_raw)
    print(alpha_th, thermal_term)
    alpha_th_interp = np.interp(r_gas, r_vcorr, alpha_th)
    thermal_term_interp = np.interp(r_gas, r_vcorr, thermal_term)
    # --- Interpolate everything onto the same x grid ---
    v_circ = interp_profile(r_vcirc, v_circ_raw, r_gas)
    v_corr_p = interp_profile(r_vcorr, v_corr_p_raw, r_gas)
    v_corr_d = interp_profile(r_vcorr, v_corr_d_raw, r_gas)
    v_corr_a = interp_profile(r_vcorr, v_corr_a_raw, r_gas)
    v_corr_t = interp_profile(r_vcorr, v_corr_t_raw, r_gas)
    # --- Radius ---
    r_01 = s.v.r01_cgas.to_value()
    r_1 = s.v.r1_cgas.to_value()
    idx_fid = np.argmin(np.abs(r_gas - r_01))

    # Extract vmax and v at fiducial radius
    for v_profile, vmax_list, vfid_list in zip([v_phi, v_corr_p, v_corr_d, v_corr_a, v_corr_t], [vmax_phi_list, vmax_corr_p_list, vmax_corr_d_list, vmax_corr_a_list, vmax_corr_t_list], [vfid_phi_list, vfid_corr_p_list, vfid_corr_d_list, vfid_corr_a_list, vfid_corr_t_list]):
        vmax = np.nanmax(v_profile)
        r_fiducial = vmax/35
        vfid = np.interp(r_fiducial, r_gas, v_profile)
        vmax_list.append(vmax)
        vfid_list.append(vfid)

    full_v_data.append({"r": r_gas, "v_phi": v_phi, "v_corr_p": v_corr_p, "v_corr_d": v_corr_d, "v_corr_a": v_corr_a, "v_corr_t": v_corr_t, "v_circ": v_circ, "beta": beta, "alpha_th": alpha_th_interp, "thermal_term": thermal_term_interp, "sigma_r": sigma_r, "sigma_phi": sigma_phi})
    # Relative errors with respect to circular velocity
    error_vphi = np.array(v_phi - v_circ)/v_circ
    error_vcorr_p = np.array(v_corr_p - v_circ)/v_circ
    error_vcorr_d = np.array(v_corr_d - v_circ)/v_circ
    error_vcorr_a = np.array(v_corr_a - v_circ)/v_circ
    error_vcorr_t = np.array(v_corr_t - v_circ)/v_circ
    r_r01 = np.linspace(0, 1, 20)
    vphi_list.append(np.interp(r_r01, r_gas / r_01, error_vphi))
    vcorr_p_list.append(np.interp(r_r01, r_gas / r_01, error_vcorr_p))
    vcorr_d_list.append(np.interp(r_r01, r_gas / r_01, error_vcorr_d))
    vcorr_a_list.append(np.interp(r_r01, r_gas / r_01, error_vcorr_a))
    vcorr_t_list.append(np.interp(r_r01, r_gas / r_01, error_vcorr_t))

    vphi_list_r1.append(np.interp(r_r01, r_gas / r_1, error_vphi) if r_1 > 1e-2 else np.full_like(r_r01, np.nan))
    vcorr_p_list_r1.append(np.interp(r_r01, r_gas / r_1, error_vcorr_p) if r_1 > 1e-2 else np.full_like(r_r01, np.nan))
    vcorr_d_list_r1.append(np.interp(r_r01, r_gas / r_1, error_vcorr_d) if r_1 > 1e-2 else np.full_like(r_r01, np.nan))
    vcorr_a_list_r1.append(np.interp(r_r01, r_gas / r_1, error_vcorr_a) if r_1 > 1e-2 else np.full_like(r_r01, np.nan))
    vcorr_t_list_r1.append(np.interp(r_r01, r_gas / r_1, error_vcorr_t) if r_1 > 1e-2 else np.full_like(r_r01, np.nan))


In [ ]:
import matplotlib.cm as cm
from matplotlib.colors import LogNorm
from matplotlib.lines import Line2D
from matplotlib.ticker import NullFormatter

fig, ax = plt.subplots(figsize=(7,6.2))
cmap = plt.get_cmap("jet")
norm = LogNorm(vmin=4e-2, vmax=2e-1)

for i, sim in enumerate(g4):
    sim_data = full_v_data[i]
    s = sim.snap[0]
    fg = s.d.mass_in_sphere(force_recompute=False, center = s.v.center, radius = s.v.r01_cgas.to_value(), particle="baryon", auto_save=True)/s.d.mass_in_sphere(force_recompute=False, center = s.v.center, radius = s.v.r01_cgas.to_value(), particle="all", auto_save=True)

    r01 = s.v.r01_cgas.to_value()
    r1 = s.v.r1_cgas.to_value()
    sim_data["r"] = np.array(sim_data["r"])
    sim_data["v_circ"] = np.array(sim_data["v_circ"])
    mask1 = sim_data["r"] < r1
    mask2 = sim_data["r"] >= r1
    mask2[np.where(mask1)[0][-1] if np.any(mask1) else None] = True  # Ensure the last point of mask1 is included in mask2
    
    #axis = s.d.faceon_weighted(force_recompute=False, radius = r01, particle="PartType0", weight="HI_mass", center = s.v.center, auto_save=True)
    #mgas = s.d.mass_in_los(force_recompute=False, center = s.v.center, radius = r01, particle="PartType0", height = s.v.r_virial.to_value(), mass_field = "Masses", axis = axis.v.tolist(), auto_save=True)
    #ms = s.d.mass_in_los(force_recompute=False, center = s.v.center, radius = r01, particle="PartType4", height = s.v.r_virial.to_value(), mass_field = "Masses", axis = axis.v.tolist(), auto_save=True)
    #surf_den = (ms+mgas)/(r01*1e6)

    vmax = np.nanmax(sim_data["v_circ"])
    rfid = vmax/35
    vfid = np.interp(rfid, sim_data["r"], sim_data["v_circ"])

    ax.plot(sim_data["r"][mask1]/r01, sim_data["v_circ"][mask1], color=cmap(norm(fg)), linewidth = 2)
    ax.scatter(rfid/r01, vfid, color=cmap(norm(fg)), edgecolor='k', s=100, zorder=5)
    ax.plot(sim_data["r"][mask2]/r01, sim_data["v_circ"][mask2], color=cmap(norm(fg)), linewidth = 2, linestyle="--")

gal_handles = [
    Line2D([], [], marker='o', linestyle='None', markersize=8,
            markerfacecolor='none', markeredgecolor="k",
            markeredgewidth=2, label=r'$V_{\rm fid}$'),
    Line2D([], [], marker='None', linestyle='-', markersize=8,
            color='k', markeredgecolor='none',
            label=r'Under $R_{\rm 1,HI}$'),
    Line2D([], [], marker='None', linestyle='--', markersize=8,
            color='k', markeredgecolor='none',
            label=r'Above $R_{\rm 1,HI}$')]
leg1 = ax.legend(handles=gal_handles, fontsize=18, loc="lower right", handletextpad=0.25, handlelength = 1)
ax.add_artist(leg1)

ax.set_xlabel(r"Radius $R/R_{\rm 0.1,HI}$", fontsize=22)
ax.set_ylabel(r"Circular Velocity $V_{\rm circ}$ [km/s]", fontsize=22)
sm = cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, pad=0.01)
cbar.set_label(r"Baryon Fraction $f_b$ within $R_{\rm 0.1, HI}$", fontsize=20)
cbar.ax.yaxis.set_minor_formatter(NullFormatter())

fig.savefig(r"C:\Home\Astro\TFG\Figures\Draft\All_Circ_Profiles.png", dpi=300)

In [ ]:
error = {}
for corr in ["v_corr_p", "v_corr_d", "v_corr_a", "v_corr_t"]:
    error[corr] = np.nanmean(np.abs(sim_data[corr] - sim_data['v_circ']) / sim_data['v_circ'])
best_corr = min(error, key=error.get)
print(f"Best correction for {sim.name}: {best_corr} with error {error[best_corr]:.2f}")

In [ ]:
import matplotlib.cm as cm
from matplotlib.colors import LogNorm
from matplotlib.lines import Line2D
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable                

def plot_sim_circ(k, idx, ax, filtered_obs = None, put_legend = False):
    sim = g4[idx]
    sim_data = full_v_data[idx]
    s = sim.snap[0]
    
    # Load sigma_phi dynamically since it wasn't appended to full_v_data
    meta_v, plot_data_v, _, _ = load_npz_data(
        rf"C:\Home\Astro\TFG\Figures\Circ_models\Circ_thermal_cilinder_half\Circ_{sim.name}.npz"
    )
    r_gas = sim_data["r"]

    r1 = s.v.r1_cgas.to_value()
    vmax = np.nanmax(sim_data["v_circ"])
    r_fid = vmax / 35.0
    sim_max_r = np.nanmax(sim_data["r"])

    error = {}
    for corr in ["v_corr_p", "v_corr_d", "v_corr_a", "v_corr_t"]:
        error[corr] = np.nanmean(np.abs(sim_data[corr] - sim_data['v_circ']) / sim_data['v_circ'])
    best_corr = min(error, key=error.get)
    print(f"Best correction for {sim.name}: {best_corr} with error {error[best_corr]:.2f}")

    # --- Plot Simulation Kinematics ---
    l1, = ax.plot(r_gas, sim_data["v_circ"], color="k", linewidth=4, label=r"$V_{\rm circ}$")
    l2, = ax.plot(r_gas, sim_data["v_phi"], color="#8ca252", linewidth=2, label=r"$V_{\phi}$")
    l3, = ax.plot(r_gas, sim_data["sigma_r"], color="#8ca252", linewidth=2, linestyle="--", label=r"$\sigma_{\rm r}$")
    l4, = ax.plot(r_gas, sim_data[best_corr], color="#ad494a", linewidth=2, label=r"$V_{\rm corr}$")
    
    # Add vertical dashed lines with nice distinct colors
    vl1 = ax.axvline(r1, color='k', linestyle='--', linewidth=1, alpha=0.8, label=r"$R_{1,\rm HI}$", zorder= -4)
    vl2 = ax.axvline(r_fid, color='k', linestyle=':', linewidth=1, alpha=0.8, label=r"$R_{\rm fid}$", zorder= -4)

    # --- Plot Observations ---
    # Filter by radius similarity: Obs max radius must be within [0.4x, 2.5x] of Sim max radius
    valid_obs =[]
    for g in observed_galaxies:
        if np.isnan(g.v_max) or len(g.rad_profile) < 8:
            continue
        obs_max_r = np.nanmax(g.rad_profile)
        if 0.7 * sim_max_r <= obs_max_r <= 1.1 * sim_max_r:
            valid_obs.append(g)
            
    # Find the 2 closest observational galaxies by v_max from the filtered list
    valid_obs.sort(key=lambda g: abs(g.v_max - vmax))
    valid_obs = [obs for obs in valid_obs if obs not in filtered_obs] if filtered_obs is not None else valid_obs
    obs_closest = valid_obs[:2]

    obs_colors = ["#1f77b4", "#ff7f0e"] # Standard blue/orange
    obs_lines = []
    for j, obs in enumerate(obs_closest):
        err = ax.errorbar(
            obs.rad_profile, obs.v_obs_profile, yerr=obs.e_v_obs_profile,
            fmt='D', markersize=6, color=obs_colors[j], alpha=0.4, zorder = -2,
            label=fr"{obs.name.upper()}"
        )
        obs_lines.append(err)

    # --- Formatting & Decorations ---
    sim_name_clean = sim.name.split('-')[-1]
    label_map = {"v_corr_p": "Pressure", "v_corr_d": "Dispersion", "v_corr_a": "Anisotropy", "v_corr_t": "Thermal"}
    ax.set_title(fr"{sim_name_clean}"+r"$-$"+rf"$\langle | \Delta V_{{\rm {label_map[best_corr]}}} | / V_{{\rm circ}} \rangle = {error[best_corr]:.2f}$", fontsize=20, pad = 14)
    ax.set_xlabel(r"Radius [kpc]", fontsize=24)
    ax.tick_params(labelsize=16)
    ax.set_ylim(0,50)

    #etarot = np.interp(r_fid,sim_data["r"],sim_data["v_circ"])/vmax
    #ax.text(0.95, 0.95, rf"$\langle | \Delta V_{{\rm {label_map[best_corr]}}} | / V_{{\rm circ}} \rangle = {error[best_corr]:.2f}$", transform=ax.transAxes, fontsize=16, va='top', ha='right')
    
    # 1. Observational legend on every axis (Upper Left)
    fs_leg = 15
    obs_leg = ax.legend(handles=obs_lines, fontsize=fs_leg, loc="upper left", framealpha=1, ncols = 2, frameon = True, fancybox=True)
    ax.add_artist(obs_leg)
    
    #rad_label = [r"$R_{\rm 1, HI}$", r"$R_{\rm fid}$"]
    #for rv, rl, rp in zip([r1, r_fid], rad_label, [0.9, 0.9]):
        #+f" = {rv:.2f} kpc"
        #ax.annotate(rl, xy=(rv, rp), xycoords=('data', 'axes fraction'), xytext=(5,0), textcoords='offset points', rotation=-90, va='top', fontsize=13, zorder = -4)

    leg_fs = 18
    if put_legend:
        divider = make_axes_locatable(ax)
        sf_leg = divider.append_axes("right", size="20%", pad=0.1)
        sf_leg.axis("off")
        sf_leg.legend(handles=[l1,l4,l2,l3,vl1,vl2], fontsize=leg_fs, loc="upper left", framealpha=1, ncols = 1, frameon = False, fancybox=False)

    return obs_closest

# 3. Populate subplots
fig, axes = plt.subplots(1, 3, figsize=(6.5*2.4, 6), sharey=False, layout="constrained", width_ratios=[1,1,1.2])

# 1. Target the specific galaxies (explicitly avoiding the AGORA variant of 1e10q)
target_names =[
    #"GADGET4-CDwarf-halo291", 
    "GADGET4-CDwarf-halo307", 
    #"GADGET4-CDwarf-halo324", 
    #"GADGET4-CDwarf-halo224"
    "GADGET4-CDwarf-halo211", 
    "GADGET4-CDwarf-1e10q",
    
]
indices =[next(i for i, sim in enumerate(g4) if sim.name == name) for name in target_names]



filtered_obs = []
for k, (idx, ax) in enumerate(zip(indices, axes)):
    extra_param = {"put_legend": True} if k == len(axes)-1 else {}
    filtered_obs.extend(plot_sim_circ(k, idx, ax, filtered_obs, **extra_param))
    
    # Only label Y-axis on the leftmost plot
    if k == 0:
        ax.set_ylabel(r"Velocity [km/s]", fontsize=22)

fig.savefig(r"C:\Home\Astro\TFG\Figures\Draft\Velocity_Profiles_Comparison_only_3.png", dpi=300)


In [ ]:
import matplotlib.cm as cm
from matplotlib.colors import LogNorm
from matplotlib.lines import Line2D
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable                

def plot_sim_circ(k, idx, ax, filtered_obs = None, put_legend = False):
    sim = g4[idx]
    sim_data = full_v_data[idx]
    s = sim.snap[0]
    
    # Load sigma_phi dynamically since it wasn't appended to full_v_data
    meta_v, plot_data_v, _, _ = load_npz_data(
        rf"C:\Home\Astro\TFG\Figures\Circ_models\Circ_thermal_cilinder_half\Circ_{sim.name}.npz"
    )
    r_gas = sim_data["r"]

    r1 = s.v.r1_cgas.to_value()
    vmax = np.nanmax(sim_data["v_circ"])
    r_fid = vmax / 35.0
    sim_max_r = np.nanmax(sim_data["r"])

    error = {}
    for corr in ["v_corr_p", "v_corr_d", "v_corr_a", "v_corr_t"]:
        error[corr] = np.nanmean(np.abs(sim_data[corr] - sim_data['v_circ']) / sim_data['v_circ'])
    best_corr = min(error, key=error.get)
    print(f"Best correction for {sim.name}: {best_corr} with error {error[best_corr]:.2f}")

    # --- Plot Simulation Kinematics ---
    l1, = ax.plot(r_gas, sim_data["v_circ"], color="k", linewidth=4, label=r"$V_{\rm circ}$")
    l2, = ax.plot(r_gas, sim_data["v_phi"], color="#8ca252", linewidth=2, label=r"$V_{\phi}$")
    l3, = ax.plot(r_gas, sim_data["sigma_r"], color="#8ca252", linewidth=2, linestyle="--", label=r"$\sigma_{\rm r}$")
    l4, = ax.plot(r_gas, sim_data[best_corr], color="#ad494a", linewidth=2, label=r"$V_{\rm corr}$")
    
    # Add vertical dashed lines with nice distinct colors
    vl1 = ax.axvline(r1, color='k', linestyle='--', linewidth=1, alpha=0.8, label=r"$R_{1,\rm HI}$", zorder= -4)
    vl2 = ax.axvline(r_fid, color='k', linestyle=':', linewidth=1, alpha=0.8, label=r"$R_{\rm fid}$", zorder= -4)

    # --- Formatting & Decorations ---
    sim_name_clean = sim.name.split('-')[-1]
    label_map = {"v_corr_p": "Pressure", "v_corr_d": "Dispersion", "v_corr_a": "Anisotropy", "v_corr_t": "Thermal"}
    #ax.set_title(fr"{sim_name_clean}", fontsize=20, pad = 14)
    ax.set_xlabel(r"Radius [kpc]", fontsize=24)
    ax.tick_params(labelsize=16)
    ax.set_ylim(0,50)

    #etarot = np.interp(r_fid,sim_data["r"],sim_data["v_circ"])/vmax
    #ax.text(0.95, 0.95, rf"$\langle | \Delta V_{{\rm {label_map[best_corr]}}} | / V_{{\rm circ}} \rangle = {error[best_corr]:.2f}$", transform=ax.transAxes, fontsize=16, va='top', ha='right')
    
    # 1. Observational legend on every axis (Upper Left)
    fs_leg = 15
    
    #rad_label = [r"$R_{\rm 1, HI}$", r"$R_{\rm fid}$"]
    #for rv, rl, rp in zip([r1, r_fid], rad_label, [0.9, 0.9]):
        #+f" = {rv:.2f} kpc"
        #ax.annotate(rl, xy=(rv, rp), xycoords=('data', 'axes fraction'), xytext=(5,0), textcoords='offset points', rotation=-90, va='top', fontsize=13, zorder = -4)

    leg_fs = 20
    if put_legend:
        divider = make_axes_locatable(ax)
        sf_leg = divider.append_axes("right", size="15%", pad=0)
        sf_leg.axis("off")
        sf_leg.legend(handles=[l1,l4,l2,l3,vl1,vl2], fontsize=leg_fs, loc="upper left", framealpha=1, ncols = 1, frameon = False, fancybox=False)

# 3. Populate subplots
fig, axes = plt.subplots(1, 1, figsize=(6, 6), sharey=False, layout="constrained")

# 1. Target the specific galaxies (explicitly avoiding the AGORA variant of 1e10q)
target_names =[
    #"GADGET4-CDwarf-halo291", 
    "GADGET4-CDwarf-halo307", 
    #"GADGET4-CDwarf-halo324", 
    #"GADGET4-CDwarf-halo224"
    "GADGET4-CDwarf-halo211", 
    "GADGET4-CDwarf-1e10q",
    
]
indices =[next(i for i, sim in enumerate(g4) if sim.name == name) for name in target_names]

for k, (idx, ax) in enumerate(zip(indices, [axes])):
    extra_param = {"put_legend": True}
    plot_sim_circ(k, idx, ax, filtered_obs, **extra_param)
    
    # Only label Y-axis on the leftmost plot
    if k == 0:
        ax.set_ylabel(r"Velocity [km/s]", fontsize=22)

#fig.savefig(r"C:\Home\Astro\TFG\Figures\Draft\Velocity_Profiles_Comparison_only_3.png", dpi=300)


In [ ]:
target_names =[
    "GADGET4-CDwarf-halo324", 
    "GADGET4-CDwarf-halo211", 
    "GADGET4-CDwarf-1e10q"
]
indices =[next(i for i, sim in enumerate(g4) if sim.name == name) for name in target_names]

# 2. Setup the figure with constrained layout
fig, axes = plt.subplots(1, 3, figsize=(20, 6.2), sharey=False, layout="constrained")

filtered_obs = []
for k, (idx, ax) in enumerate(zip(indices, axes)):
    filtered_obs.extend(plot_sim_circ(k, idx, ax, filtered_obs))
    
    # Only label Y-axis on the leftmost plot
    if k == 0:
        ax.set_ylabel(r"Velocity [km/s]", fontsize=22)

fig.savefig(r"C:\Home\Astro\TFG\Figures\Draft\Velocity_Profiles_Comparison_2.png", dpi=300)

In [ ]:
# Convert to matrices: shape = (N_sim, N_radius)
vphi_mat = np.asarray(vphi_list)
vcorr_p_mat = np.asarray(vcorr_p_list)
vcorr_t_mat = np.asarray(vcorr_t_list)
vcorr_d_mat = np.asarray(vcorr_d_list)
vcorr_a_mat = np.asarray(vcorr_a_list)

# Median and dispersion at each radius
vphi_med = np.nanmedian(vphi_mat[1:], axis=0)
vphi_std = np.nanstd(vphi_mat[1:], axis=0)

vcorr_p_med = np.nanmedian(vcorr_p_mat[1:], axis=0)
vcorr_p_std = np.nanstd(vcorr_p_mat[1:], axis=0)

vcorr_d_med = np.nanmedian(vcorr_d_mat[1:], axis=0)
vcorr_d_std = np.nanstd(vcorr_d_mat[1:], axis=0)

vcorr_a_med = np.nanmedian(vcorr_a_mat[1:], axis=0)
vcorr_a_std = np.nanstd(vcorr_a_mat[1:], axis=0)

vcorr_t_med = np.nanmedian(vcorr_t_mat[1:], axis=0)
vcorr_t_std = np.nanstd(vcorr_t_mat[1:], axis=0)

# Plot
fig, ax = plt.subplots(1, 5, figsize=(6.5*2.4, 6)) #5.5, 8

colors = ["#8ca252", "#ad494a", "#52a295", "#6652a2","#a28252"]
def plot_med_std(ax, x, med, std, full, label, yticklabels=False):
    color = colors.pop(0)
    ax.plot(x, med, lw=2, label=label, color=color)
    ax.fill_between(x, med - std, med + std, alpha=0.25, color=color)
    ax.legend(frameon=False, loc="lower right", fontsize = 20)
    if not yticklabels:
        ax.set_yticklabels([])
    ax.set_ylim(-2.2,2.2)
    ax.axhline(0, color="k", ls=":", lw=2, zorder = -1)
    for i, sim in enumerate(g4):
        if i == 0:
            ax.plot(x, full[i, :], lw=1, alpha = 0.5, color=color, zorder = -2, linestyle="--")
            continue
        #xp = sim.snap[0].v.r1_cgas.to_value()/sim.snap[0].v.r01_cgas.to_value()
        ax.plot(x, full[i, :], lw=1, alpha = 0.3, color=color, zorder = -2)
        #ax.axvline(xp, color="magenta", ls="--", lw=2, zorder = -1)
    ax.set_xlim(0,0.9)
    ax.set_xlabel(r"$R/R_{\rm 0.1,HI}$", fontsize=24)

    bias = np.nanmean(med)
    scatter = np.nanmean(std)
    text_str = (
        r"$\begin{aligned}"
        r"\langle r \rangle  &= " + f"{bias:.2f}" + r" \\" 
        r"\sigma  &= " + f"{scatter:.2f}" + r" \\" 
        r"\end{aligned}$")
    ax.text(0.95, 0.95, text_str, transform=ax.transAxes, fontsize=16, va='top', ha='right')
    ax.tick_params(labelsize=16)


plot_med_std(ax[0], r_r01, vphi_med,     vphi_std, vphi_mat,     r"$V_{\phi}$", yticklabels=True)
plot_med_std(ax[1], r_r01, vcorr_p_med,   vcorr_p_std,  vcorr_p_mat,  r"$\mathrm{Pressure}$")
plot_med_std(ax[2], r_r01, vcorr_t_med,   vcorr_t_std,  vcorr_t_mat,  r"$\mathrm{Thermal}$")
plot_med_std(ax[3], r_r01, vcorr_d_med,   vcorr_d_std,  vcorr_d_mat,  r"$\mathrm{Dispersion}$")
plot_med_std(ax[4], r_r01, vcorr_a_med,   vcorr_a_std,  vcorr_a_mat,  r"$\mathrm{Anisotropy}$")

ax[0].set_ylabel(r"$\left(V_{\rm rot}-V_{\rm circ}\right)/V_{\rm circ}$", fontsize = 24)

fig.savefig(r"C:\Home\Astro\TFG\Figures\Draft\Correction_accuracy.png", dpi=300)

In [ ]:
# Convert to matrices: shape = (N_sim, N_radius)
vphi_mat = np.asarray(vphi_list)
vcorr_p_mat = np.asarray(vcorr_p_list)
vcorr_t_mat = np.asarray(vcorr_t_list)
vcorr_d_mat = np.asarray(vcorr_d_list)
vcorr_a_mat = np.asarray(vcorr_a_list)

# Median and dispersion at each radius
vphi_med = np.nanmedian(vphi_mat[1:], axis=0)
vphi_std = np.nanstd(vphi_mat[1:], axis=0)

vcorr_p_med = np.nanmedian(vcorr_p_mat[1:], axis=0)
vcorr_p_std = np.nanstd(vcorr_p_mat[1:], axis=0)

vcorr_d_med = np.nanmedian(vcorr_d_mat[1:], axis=0)
vcorr_d_std = np.nanstd(vcorr_d_mat[1:], axis=0)

vcorr_a_med = np.nanmedian(vcorr_a_mat[1:], axis=0)
vcorr_a_std = np.nanstd(vcorr_a_mat[1:], axis=0)

vcorr_t_med = np.nanmedian(vcorr_t_mat[1:], axis=0)
vcorr_t_std = np.nanstd(vcorr_t_mat[1:], axis=0)

# Plot
fig, ax = plt.subplots(1, 2, figsize=(6.5*2, 6)) #5.5, 8

colors = ["#8ca252", "#ad494a", "#52a295", "#6652a2","#a28252"]
def plot_med_std(ax, x, med, std, full, label, yticklabels=False):
    color = colors.pop(0)
    ax.plot(x, med, lw=2, label=label, color=color)
    ax.fill_between(x, med - std, med + std, alpha=0.25, color=color)
    ax.legend(frameon=False, loc="lower right", fontsize = 20)
    if not yticklabels:
        ax.set_yticklabels([])
    ax.set_ylim(-2.2,2.2)
    ax.axhline(0, color="k", ls=":", lw=2, zorder = -1)
    for i, sim in enumerate(g4):
        if i == 0:
            ax.plot(x, full[i, :], lw=1, alpha = 0.5, color=color, zorder = -2, linestyle="--")
            continue
        #xp = sim.snap[0].v.r1_cgas.to_value()/sim.snap[0].v.r01_cgas.to_value()
        ax.plot(x, full[i, :], lw=1, alpha = 0.3, color=color, zorder = -2)
        #ax.axvline(xp, color="magenta", ls="--", lw=2, zorder = -1)
    ax.set_xlim(0,0.9)
    ax.set_xlabel(r"$R/R_{\rm 0.1,HI}$", fontsize=24)

    bias = np.nanmean(med)
    scatter = np.nanmean(std)
    text_str = (
        r"$\begin{aligned}"
        r"\langle r \rangle  &= " + f"{bias:.2f}" + r" \\" 
        r"\sigma  &= " + f"{scatter:.2f}" + r" \\" 
        r"\end{aligned}$")
    ax.text(0.95, 0.95, text_str, transform=ax.transAxes, fontsize=16, va='top', ha='right')
    ax.tick_params(labelsize=16)


plot_med_std(ax[0], r_r01, vphi_med,     vphi_std, vphi_mat,     r"$V_{\phi}$", yticklabels=True)
plot_med_std(ax[1], r_r01, vcorr_p_med,   vcorr_p_std,  vcorr_p_mat,  r"$\mathrm{Pressure}$")
#plot_med_std(ax[2], r_r01, vcorr_t_med,   vcorr_t_std,  vcorr_t_mat,  r"$\mathrm{Thermal}$")
#plot_med_std(ax[3], r_r01, vcorr_d_med,   vcorr_d_std,  vcorr_d_mat,  r"$\mathrm{Dispersion}$")
#plot_med_std(ax[4], r_r01, vcorr_a_med,   vcorr_a_std,  vcorr_a_mat,  r"$\mathrm{Anisotropy}$")

ax[0].set_ylabel(r"$\left(V_{\rm rot}-V_{\rm circ}\right)/V_{\rm circ}$", fontsize = 24)

#fig.savefig(r"C:\Home\Astro\TFG\Figures\Draft\Correction_accuracy.png", dpi=300)

In [ ]:
# Load the prev snapshot and compute the rotation curves. Also compute the mass loss per stellar particle (tracking IDs) and also consider the "mass gain" for stars that have been formed. Store the position each event happened. Try to correlate this mass loss/gain with dispersion, pressure and density profiles, which in the end are the causes of the corrections to the rotation curve. Also correlate to the change in the profiles between the two snapshots (this should answer why the profiles are noisy)

In [ ]:
fig, ax = plt.subplots(figsize=(8,6))

for i,sim in enumerate(g4):
    ds = sim[0]
    s = sim.snap[0]
    sfr = s.d.sfr_young_star(force_recompute=False, center = s.v.center, radius = s.v.r01_cgas.to_value()/20, max_age = 100, auto_save=False, label = "sfr_young_star")

    corr = np.asarray(vcorr_p_list)[0,i]

    ax.plot(sfr,corr, 'o', label=sim.name)

    print(corr,sfr)

ax.legend()

In [ ]:
fig, ax = plt.subplots(5,3,figsize=(8*3,6*5),sharex=True)
for i,sim in enumerate(g4):
    ds = sim[0]
    s = sim.snap[0]

    sp = ds.sphere(s.v.center, (s.v.r_virial.to_value(), "kpc"))

    data_args = af.settings.DataConfig(log=False, accumulate=False, unit="Msun/yr", nbins = 200,
    postprocess="sfh", postprocess_kwargs={"time_unit": "Myr","as_lookback": True,})

    style_args = af.settings.StyleConfig(title=r"SFH ($r < r_{\rm virial}$)", xlabel="Lookback Time (Myr)", ylabel="SFR (Msun/yr)", linewidth=2.0, grid=True, ax = ax.flatten()[i], label = sim.name.split("-")[-1], log_y = True)

    io_args = af.settings.IOConfig(show=True, return_fig=True, save=False, return_data=False)

    af.plot.profile(sp, ("PartType4", "StellarFormationTime"), ("PartType4", "Masses"), data_args=data_args,style_args=style_args, io_args=io_args)
    ax.flatten()[i].legend(ncol=3)

In [ ]:
#g4[0][0].derived_field_list

#StarFormationRate, nStarSpawn, SNIIFeedbackFlag,AGBFeedbackFlag,SNIaFeedbackFlag

s = g4[0].snap[0]
ds = g4[0][0].sphere(s.v.center, (s.v.r_virial.to_value(), "kpc"))

sfr = ds[("PartType0", "StarFormationRate")].sum()
stspawn = ds[("PartType0", "nStarSpawn")].sum()
fbflag1 = ds[("PartType4", "SNIIFeedbackFlag")].sum()
fbflag2 = ds[("PartType4", "SNIaFeedbackFlag")].sum()
fbflag3 = ds[("PartType4", "AGBFeedbackFlag")].sum()

print(f"Total SFR: {sfr:.2e}")
print(f"Total Star Formation: {stspawn:.2e}")
print(f"Total SNIa Feedback: {fbflag2:.2e}")
print(f"Total SNII Feedback: {fbflag1:.2e}")
print(f"Total AGB Feedback: {fbflag3:.2e}")

def _FB_Event(field, data):
    return data[("PartType4", "SNIIFeedbackFlag")] + data[("PartType4", "SNIaFeedbackFlag")] + data[("PartType4", "AGBFeedbackFlag")]

for sim in g4:
    sim[0].add_field(("PartType4", "FB_Event"), function=_FB_Event, units="dimensionless", display_name="Feedback Events", sampling_type="particle", force_override=True, take_log = False)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, ax = plt.subplots(5, 3, figsize=(8*3, 6*5))
axes = ax.flatten()

def r01_rad(s, altsim = None):
    sim_to_use = s if altsim is None else altsim
    return sim_to_use.v.r01_cgas.to_value()

for i, sim in enumerate(g4):
    ds = sim[0]
    s = sim.snap[0]

    # 1. Get the disk region
    disk = gen_sel_disk_fn(z_fn = lambda s: s.v.h_e_HI.to_value(), radius_fn = r01_rad, set_vbulk = True)
    sp = disk(s)

    # 2. Get Stellar Formation Times (Scale Factor) and compute ages
    a_form = sp[("PartType4", "StellarFormationTime")].value
    z_form = 1.0 / np.clip(a_form, 1e-8, 1.0) - 1.0
    
    # Cosmology conversion to Lookback Time
    t_form = ds.cosmology.t_from_z(z_form).to("Myr").value
    t_now = ds.current_time.to("Myr").value
    ages = t_now - t_form
    
    # 3. Get total Feedback Events per particle
    fb_events = (sp[("PartType4", "SNIIFeedbackFlag")] + 
                 sp[("PartType4", "SNIaFeedbackFlag")] + 
                 sp[("PartType4", "AGBFeedbackFlag")]).value
    
    # 4. Filter for RECENT feedback (e.g., last 100 Myr)
    # This ignores ancient stars whose feedback dissipated billions of years ago
    recent_mask = ages < 50 
    recent_fb = fb_events[recent_mask]
    
    # Get Radii in kpc
    radii = sp[("PartType4", "particle_position_cylindrical_radius")].to("kpc").value
    recent_radii = radii[recent_mask]
    
    # 5. Bin into radial profile
    r_max = s.v.r01_cgas.to_value()
    bins = np.linspace(0, r_max, 50)
    bin_centers = (bins[:-1] + bins[1:]) / 2
    
    # Sum the recent feedback events in each bin
    fb_in_bins, _ = np.histogram(recent_radii, bins=bins, weights=recent_fb)
    
    # Calculate area of each annulus to get Surface Density of Events
    area_bins = np.pi * (bins[1:]**2 - bins[:-1]**2)
    sigma_fb = fb_in_bins / area_bins
    
    # 6. Plotting
    axes[i].plot(bin_centers, sigma_fb, lw=2, color="#ad494a", label=sim.name.split("-")[-1])
    axes[i].fill_between(bin_centers, 0, sigma_fb, color="#ad494a", alpha=0.3)
    
    axes[i].set_xlabel("Radius (kpc)", fontsize=12)
    axes[i].set_ylabel(r"$\Sigma_{\rm FB \ (< 100 \ Myr)}$[Events / kpc$^2$]", fontsize=12)
    axes[i].set_xlim(0, r_max)
    axes[i].grid(alpha=0.4, ls="--")
    axes[i].legend(loc="upper right")

# Hide any unused subplots
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

def plot_feedback_error_correlation(g4, full_v_data, models_to_plot=["v_phi", "v_corr_a"], lookback_myr=50):
    """
    Plots the correlation between local recent feedback surface density and 
    the relative error of chosen velocity models.
    
    Parameters:
    - g4: list of simulation objects
    - full_v_data: list of dicts containing the kinematic profiles for each sim
    - models_to_plot: list of keys from full_v_data to plot and compare
    - lookback_myr: maximum age of stars to consider for recent feedback
    """
    
    # Dictionary to hold the paired data across all simulations
    collected_data = {model: {"sfb": [], "err":[]} for model in models_to_plot}
    
    # Map model keys to readable titles
    title_map = {
        "v_phi": r"Uncorrected $V_\phi$",
        "v_corr_p": r"Pressure Corrected",
        "v_corr_d": r"Non-cte Disp. Corrected",
        "v_corr_a": r"Anisotropy Corrected"
    }

    print("Extracting and matching feedback events to kinematic grids...")
    for i, sim in enumerate(g4):
        ds = sim[0]
        s = sim.snap[0]
        v_data = full_v_data[i]
        r_gas = v_data["r"]
        v_circ = v_data["v_circ"]
        
        # 1. Build bin edges from the r_gas centers
        dr = np.diff(r_gas)
        if len(dr) == 0: continue # Skip if only 1 point
        edges = np.zeros(len(r_gas) + 1)
        edges[1:-1] = r_gas[:-1] + dr / 2.0
        edges[0] = max(0, r_gas[0] - dr[0] / 2.0)
        edges[-1] = r_gas[-1] + dr[-1] / 2.0
        
        # 2. Extract recent feedback events
        sp = ds.sphere(s.v.center, (s.v.r_virial.to_value(), "kpc"))
        a_form = sp[("PartType4", "StellarFormationTime")].value
        z_form = 1.0 / np.clip(a_form, 1e-8, 1.0) - 1.0
        
        t_form = ds.cosmology.t_from_z(z_form).to("Myr").value
        t_now = ds.current_time.to("Myr").value
        ages = t_now - t_form
        
        fb_events = (sp[("PartType4", "SNIIFeedbackFlag")] + 
                     sp[("PartType4", "SNIaFeedbackFlag")] + 
                     sp[("PartType4", "AGBFeedbackFlag")]).value
                     
        recent_mask = ages < lookback_myr
        recent_fb = fb_events[recent_mask]
        
        radii = sp[("PartType4", "particle_position_cylindrical_radius")].to("kpc").value
        recent_radii = radii[recent_mask]
        
        # 3. Bin events onto the r_gas grid
        fb_in_bins, _ = np.histogram(recent_radii, bins=edges, weights=recent_fb)
        area_bins = np.pi * (edges[1:]**2 - edges[:-1]**2)
        sigma_fb = fb_in_bins / area_bins
        
        # 4. Compute errors for chosen models and store
        for model in models_to_plot:
            #v_model = v_data[model]
            #with np.errstate(divide='ignore', invalid='ignore'):
            #    err = np.abs((v_model - v_circ) / v_circ)
                
            err = v_data["sigma_r"]

            # Keep only valid finite numbers
            valid = np.isfinite(err) & np.isfinite(sigma_fb) & (sigma_fb >= 1e-2)
            collected_data[model]["sfb"].extend(sigma_fb[valid])
            collected_data[model]["err"].extend(err[valid])

    # --- Plotting ---
    n_models = len(models_to_plot)
    fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 6), sharey=True)
    if n_models == 1: axes = [axes]
    
    # Floor to allow plotting 0-feedback bins on a log scale safely
    fb_floor = 1e-2 
    colors =["#8ca252", "#ad494a", "#52a295", "#6652a2"]
    
    def plot_running_median(ax, x, y, color):
        bins = np.logspace(np.log10(fb_floor), np.log10(np.max(x)*1.1), 12)
        bin_centers = (bins[:-1] + bins[1:]) / 2
        medians, p25, p75 = np.zeros_like(bin_centers), np.zeros_like(bin_centers), np.zeros_like(bin_centers)
        
        for k in range(len(bins)-1):
            mask = (x >= bins[k]) & (x < bins[k+1])
            if np.sum(mask) > 3:
                medians[k] = np.nanmedian(y[mask])
                p25[k] = np.nanpercentile(y[mask], 25)
                p75[k] = np.nanpercentile(y[mask], 75)
            else:
                medians[k], p25[k], p75[k] = np.nan, np.nan, np.nan
                
        ax.plot(bin_centers, medians, color=color, lw=3, label="Median Trend")
        ax.fill_between(bin_centers, p25, p75, color=color, alpha=0.3)

    for idx, model in enumerate(models_to_plot):
        ax = axes[idx]
        color = colors[idx % len(colors)]
        
        x_data = np.array(collected_data[model]["sfb"])
        y_data = np.array(collected_data[model]["err"])
        
        # Calculate Spearman Correlation
        stat, pval = spearmanr(x_data, y_data)
        
        # Clip zeros for log plotting
        
        ax.scatter(x_data, y_data, alpha=0.35, color=color, edgecolors='none', s=40)
        plot_running_median(ax, x_data, y_data, "black")
        
        ax.set_xscale("log")
        ax.set_xlabel(r"Local $\Sigma_{\rm FB \ (< 100 \ Myr)}$ $[\rm{Events} / \rm{kpc}^2]$", fontsize=14)
        ax.set_xlim(fb_floor * 0.8, np.max(x_data)*1.5)
        #ax.set_ylim(0, 1.5)  # Cap visually at 150% error, adjust if needed
        
        # Mark the "zero" floor
        ax.axvline(fb_floor, color='gray', linestyle='--', alpha=0.6, zorder=-1)
        ax.text(fb_floor*1.1, 1.4, "Zero\nFeedback", color='gray', fontsize=11, va='top')
        
        # Add Statistics Box
        stat_text = (
            r"$\mathbf{Spearman \ \rho:}$" + f"\n"
            f"$r_s = {stat:.2f}$\n"
            f"$p = {pval:.1e}$"
        )
        props = dict(boxstyle='round', facecolor='white', alpha=0.8, edgecolor='gray')
        ax.text(0.05, 0.95, stat_text, transform=ax.transAxes, fontsize=12,
                verticalalignment='top', bbox=props)
        
        ax.set_title(title_map.get(model, model), fontsize=16)
        if idx == 0:
            ax.set_ylabel(r"Relative Error $|V_{\rm model} - V_{\rm circ}| / V_{\rm circ}$", fontsize=14)
            
        ax.legend(loc="upper right", frameon=False)

    plt.show()

# --- To use the function ---
# Pass your g4 array, the full_v_data dictionary list, and choose which models to plot
plot_feedback_error_correlation(g4, full_v_data, models_to_plot=["v_phi", "v_corr_a", "v_corr_p","v_corr_t"])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

def plot_radial_error_feedback(g4, full_v_data, models_to_plot=["v_phi", "v_corr_t", "v_corr_p"], lookback_myr=500):
    
    # Dictionary to hold the paired data
    collected_data = {model: {"r_norm": [], "err": [], "sfb":[]} for model in models_to_plot}
    
    title_map = {
        "v_phi": r"Uncorrected $V_\phi$",
        "v_corr_p": r"Pressure Corrected",
        "v_corr_d": r"Non-cte Disp. Corrected",
        "v_corr_a": r"Anisotropy Corrected",
        "v_corr_t": r"Thermal Corrected"
    }

    print("Matching feedback events to kinematic grids...")
    for i, sim in enumerate(g4):
        ds = sim[0]
        s = sim.snap[0]
        v_data = full_v_data[i]
        
        r_gas = v_data["r"]
        v_circ = v_data["v_circ"]
        
        # 1. Normalize Radius
        r01 = s.v.r01_cgas.to_value()
        r_norm = r_gas / r01
        
        # 2. Build bin edges from r_gas
        dr = np.diff(r_gas)
        if len(dr) == 0: continue
        edges = np.zeros(len(r_gas) + 1)
        edges[1:-1] = r_gas[:-1] + dr / 2.0
        edges[0] = max(0, r_gas[0] - dr[0] / 2.0)
        edges[-1] = r_gas[-1] + dr[-1] / 2.0
        
        # 3. Extract recent feedback events
        sp = ds.sphere(s.v.center, (s.v.r_virial.to_value(), "kpc"))
        a_form = sp[("PartType4", "StellarFormationTime")].value
        z_form = 1.0 / np.clip(a_form, 1e-8, 1.0) - 1.0
        
        t_form = ds.cosmology.t_from_z(z_form).to("Myr").value
        t_now = ds.current_time.to("Myr").value
        ages = t_now - t_form
        
        fb_events = (sp[("PartType4", "SNIIFeedbackFlag")] + 
                     sp[("PartType4", "SNIaFeedbackFlag")] + 
                     sp[("PartType4", "AGBFeedbackFlag")]).value
                     
        recent_mask = ages < lookback_myr
        recent_fb = fb_events[recent_mask]
        
        radii = sp[("PartType4", "particle_position_cylindrical_radius")].to("kpc").value
        recent_radii = radii[recent_mask]
        
        # 4. Bin onto radial grid
        fb_in_bins, _ = np.histogram(recent_radii, bins=edges, weights=recent_fb)
        area_bins = np.pi * (edges[1:]**2 - edges[:-1]**2)
        sigma_fb = fb_in_bins / area_bins
        
        # 5. Extract errors
        for model in models_to_plot:
            v_model = v_data[model]
            with np.errstate(divide='ignore', invalid='ignore'):
                err = np.abs((v_model - v_circ) / v_circ)
                
            # Filter valid
            valid = np.isfinite(err) & np.isfinite(sigma_fb) & np.isfinite(r_norm)
            collected_data[model]["r_norm"].extend(r_norm[valid])
            collected_data[model]["err"].extend(err[valid])
            collected_data[model]["sfb"].extend(sigma_fb[valid])

    # --- Plotting ---
    n_models = len(models_to_plot)
    fig, axes = plt.subplots(1, n_models, figsize=(6.5 * n_models, 6), sharey=True)
    if n_models == 1: axes = [axes]
    
    # Floor to handle 0-feedback bins in a Log colormap
    fb_floor = 1e-1 
    
    for idx, model in enumerate(models_to_plot):
        ax = axes[idx]
        
        x_data = np.array(collected_data[model]["r_norm"])
        y_data = np.array(collected_data[model]["err"])
        c_data = np.array(collected_data[model]["sfb"])
        
        # --- CRITICAL STEP: SORTING ---
        # Sort data so points with the highest feedback are plotted LAST (on top)
        # This prevents the massive number of zero-feedback points from obscuring the active ones.
        sort_idx = np.argsort(c_data)
        x_data = x_data[sort_idx]
        y_data = y_data[sort_idx]
        c_data = c_data[sort_idx]
        
        # Clip color data for the LogNorm colormap
        c_plot = np.clip(c_data, fb_floor, None)
        
        # Scatter Plot
        # Using plasma colormap: zero-feedback will be dark blue/purple, high-feedback will be bright yellow
        sc = ax.scatter(x_data, y_data, c=c_plot, 
                        cmap='plasma', 
                        norm=mcolors.LogNorm(vmin=fb_floor, vmax=np.percentile(c_plot, 99)),
                        alpha=0.7, edgecolors='none', s=40)
        
        # Plot moving median to show the general 1/r decay
        bins = np.linspace(0, 1.2, 12)
        bin_centers = (bins[:-1] + bins[1:]) / 2
        medians = np.zeros_like(bin_centers)
        
        for k in range(len(bins)-1):
            mask = (x_data >= bins[k]) & (x_data < bins[k+1])
            if np.sum(mask) > 3:
                medians[k] = np.nanmedian(y_data[mask])
            else:
                medians[k] = np.nan
                
        ax.plot(bin_centers, medians, color='black', lw=3, label="Median Error", zorder=10)
        
        # Formatting
        ax.set_title(title_map.get(model, model), fontsize=16)
        ax.set_xlabel(r"$R/R_{0.1, \rm{HI}}$", fontsize=14)
        ax.set_xlim(0, 1.0)
        ax.set_ylim(-0.05, 1.5) # Cap visually at 150% error
        
        ax.axhline(0, color='gray', linestyle='--', alpha=0.6, zorder=-1)
        ax.grid(alpha=0.3, ls=':')
        ax.legend(loc="upper right", frameon=False)
        
        if idx == 0:
            ax.set_ylabel(r"Relative Error $|V_{\rm model} - V_{\rm circ}| / V_{\rm circ}$", fontsize=14)
            
        # Colorbar
        cbar = fig.colorbar(sc, ax=ax, pad=0.01)
        cbar.set_label(r"Local $\Sigma_{\rm FB \ (< 100 \ Myr)}$ $[\rm{Events} / \rm{kpc}^2]$", fontsize=13)

    plt.show()

# Run the function
plot_radial_error_feedback(g4, full_v_data, models_to_plot=["v_phi", "v_corr_t", "v_corr_p"])

In [ ]:
# Now the same but cutoff at R_1

# Convert to matrices: shape = (N_sim, N_radius)
vphi_mat = np.asarray(vphi_list_r1)
vcorr_p_mat = np.asarray(vcorr_p_list_r1)
vcorr_d_mat = np.asarray(vcorr_d_list_r1)
vcorr_a_mat = np.asarray(vcorr_a_list_r1)
vcorr_e_mat = np.asarray(vcorr_e_list_r1)

# Median and dispersion at each radius
vphi_med = np.nanmedian(vphi_mat, axis=0)
vphi_std = np.nanstd(vphi_mat, axis=0)

vcorr_p_med = np.nanmedian(vcorr_p_mat, axis=0)
vcorr_p_std = np.nanstd(vcorr_p_mat, axis=0)

vcorr_d_med = np.nanmedian(vcorr_d_mat, axis=0)
vcorr_d_std = np.nanstd(vcorr_d_mat, axis=0)

vcorr_a_med = np.nanmedian(vcorr_a_mat, axis=0)
vcorr_a_std = np.nanstd(vcorr_a_mat, axis=0)

vcorr_e_med = np.nanmedian(vcorr_e_mat, axis=0)
vcorr_e_std = np.nanstd(vcorr_e_mat, axis=0)

# Plot
fig, ax = plt.subplots(1, 5, figsize=(5*5, 5))

colors = ["#8ca252", "#ad494a", "#52a295", "#6652a2","#a28252"]
def plot_med_std(ax, x, med, std, full, label, yticklabels=False):
    color = colors.pop(0)
    ax.plot(x, med, lw=2, label=label, color=color)
    ax.fill_between(x, med - std, med + std, alpha=0.25, color=color)
    ax.legend(frameon=False)
    if not yticklabels:
        ax.set_yticklabels([])
    ax.set_ylim(-2,2)
    ax.axhline(0, color="k", ls=":", lw=2, zorder = -1)
    for i, sim in enumerate(g4):
        #xp = sim.snap[0].v.r1_cgas.to_value()/sim.snap[0].v.r01_cgas.to_value()
        ax.plot(x, vphi_mat[i, :], lw=1, alpha = 0.2, color=color, zorder = -2)
        #ax.axvline(xp, color="magenta", ls="--", lw=2, zorder = -1)
    ax.set_xlim(0,0.9)
    ax.set_xlabel(r"$R/R_{\rm 1,HI}$")

plot_med_std(ax[0], r_r01, vphi_med,     vphi_std, vphi_mat,     r"$V_{\phi}$", yticklabels=True)
plot_med_std(ax[1], r_r01, vcorr_p_med,   vcorr_p_std,  vcorr_p_mat,  r"$\mathrm{W/ Pressure}$")
plot_med_std(ax[2], r_r01, vcorr_d_med,   vcorr_d_std,  vcorr_d_mat,  r"$\mathrm{W/ Non\text{-}cte\ Disp.}$")
plot_med_std(ax[3], r_r01, vcorr_a_med,   vcorr_a_std,  vcorr_a_mat,  r"$\mathrm{W/ Anisotropy}$")
plot_med_std(ax[4], r_r01, vcorr_e_med,   vcorr_e_std,  vcorr_e_mat,  r"$\mathrm{Euler } V_c$")
ax[0].set_ylabel(r"$\left(V_{\rm corr}-V_{\rm circ}\right)/V_{\rm circ}$")
# TODO: Put mean of residuals on text
plt.show()

In [ ]:
# If you want to restrict the radial range used for the summaries:
mask = (r_r01 >= 0.0) & (r_r01 <= 1.0)   # adjust as needed

methods = {
    r"$V_{\phi}$": vphi_mat,
    r"$\mathrm{W/ Pressure}$": vcorr_p_mat,
    r"$\mathrm{W/ Non\text{-}cte\ Disp.}$": vcorr_d_mat,
    r"$\mathrm{W/ Anisotropy}$": vcorr_a_mat,
}

for function in [np.nanmedian]: #np.nanmean,
    bias_by_method = {}
    mae_by_method = {}
    for name, mat in methods.items():
        # one scalar per simulation
        bias = function(mat[:, mask], axis=1)
        mae  = function(np.abs(mat[:, mask]), axis=1)

        bias_by_method[name] = bias
        mae_by_method[name] = mae

        print(f"\n{name}")
        for i, (b, a) in enumerate(zip(bias, mae)):
            print(f"  sim {i:02d}: bias = {b:+.4f}, abs err = {a:.4f}")

        print(f"  summary bias: mean={np.nanmean(bias):+.4f}, std={np.nanstd(bias):.4f}, median={np.nanmedian(bias):+.4f}")
        print(f"  summary abs : mean={np.nanmean(mae):+.4f}, std={np.nanstd(mae):.4f}, median={np.nanmedian(mae):+.4f}")

    labels = list(methods.keys())
    x = np.arange(len(labels))

    bias_data = [bias_by_method[k] for k in labels]
    mae_data  = [mae_by_method[k]  for k in labels]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)

    for ax, data, ylabel, title in [
        (axes[0], bias_data, r"$\langle (V_{\rm corr}-V_{\rm circ})/V_{\rm circ}\rangle$", "Bias"),
        (axes[1], mae_data,  r"$\langle |(V_{\rm corr}-V_{\rm circ})/V_{\rm circ}| \rangle$", "Absolute error"),
    ]:
        ax.boxplot(data, tick_labels=labels, showfliers=False)
        for j, vals in enumerate(data, start=1):
            ax.scatter(np.full(len(vals), j), vals, s=18, alpha=0.7)
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.tick_params(axis="x", rotation=25)
        ax.axhline(0, color="k", ls=":", lw=2, zorder = -1)
    fig.suptitle("Using radius mean" if function == np.nanmean else "Using radius median")
    plt.show()

In [ ]:
from ltsfit.ltsfit import ltsfit

lt_usable = [g for g in lt_galaxies if not np.isnan(g.eta_rot) and g.v_max < 500 and not np.isnan(g.m_bar)]
sparc_gals_usable = [g for g in sparc_galaxies if not np.isnan(g.eta_rot) and g.inclination > 30 and g.v_max < 500 and not np.isnan(g.m_bar)]

def kill_nan(arr):
    mask = ~np.isnan(arr)
    return mask

def clean_btfr(vmax, mbar, e_vmax, e_mbar):
    vmax = np.array(vmax)
    mbar = np.array(mbar)
    e_vmax = np.array(e_vmax)
    e_mbar = np.array(e_mbar)
    log_vmax   = np.log10(vmax)
    log_mbar   = np.log10(mbar)
    log_e_vmax = 1/(vmax * np.log(10)) * e_vmax
    log_e_mbar = 1/(mbar * np.log(10)) * e_mbar
    mask1 = kill_nan(log_e_vmax)
    mask2 = kill_nan(log_e_mbar)
    log_e_vmax = log_e_vmax[mask1 & mask2]
    log_e_mbar = log_e_mbar[mask1 & mask2]
    log_vmax = log_vmax[mask1 & mask2]
    log_mbar = log_mbar[mask1 & mask2]
    return log_vmax, log_mbar, log_e_vmax, log_e_mbar

lt_vmax, lt_mbar, lt_e_vmax, lt_e_mbar = clean_btfr([g.v_max for g in lt_usable], [g.m_bar for g in lt_usable], [g.e_v_max for g in lt_usable], [g.e_m_bar for g in lt_usable])
sp_vmax, sp_mbar, sp_e_vmax, sp_e_mbar = clean_btfr([g.v_max for g in sparc_gals_usable], [g.m_bar for g in sparc_gals_usable], [g.e_v_max for g in sparc_gals_usable], [g.e_m_bar for g in sparc_gals_usable])

print(sp_mbar, sp_vmax, sp_e_mbar, sp_e_vmax)
print(lt_mbar, lt_vmax, lt_e_mbar, lt_e_vmax)

plt.figure()
p1 = ltsfit(sp_vmax, sp_mbar, sp_e_vmax, sp_e_mbar, clip=10, corr=True, epsy=True,
            frac=None, label='Fitted', label_clip='Clipped',
            legend=True, pivot=None, plot=False, text=True)

plt.figure()
p2 = ltsfit(lt_vmax, lt_mbar, lt_e_vmax, lt_e_mbar, clip=10, corr=True, epsy=True,
           frac=None, label='Fitted', label_clip='Clipped',
           legend=True, pivot=None, plot=False, text=True)

plt.figure()
p3 = ltsfit(
    np.concatenate([lt_vmax, sp_vmax]),
    np.concatenate([lt_mbar, sp_mbar]),
    np.concatenate([lt_e_vmax, sp_e_vmax]),
    np.concatenate([lt_e_mbar, sp_e_mbar]),
    clip=10, corr=True, epsy=True,
    frac=None, label='Fitted', label_clip='Clipped',
    legend=True, pivot=None, plot=False, text=True) #np.median(np.concatenate([lt_vmax, sp_vmax]), 0)

print(p1.sig_int, p1.sig_int_err)
print(p2.sig_int, p2.sig_int_err)
print(p3.sig_int, p3.sig_int_err)

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

sim_colors = [
    "#1f77b4",  # blue
    "#ff7f0e",  # orange
    "#2ca02c",  # green
    "#d62728",  # red
    "#9467bd",  # purple
    "#8c564b",  # brown
    "#e377c2",  # pink
    "#7f7f7f",  # gray
    "#bcbd22",  # olive
    "#17becf",  # cyan
    "#aec7e8",  # light blue
    "#ffbb78",  # light orange
    "#98df8a",  # light green
    "#ff9896",  # light red
]
sim_handles = [
    Line2D([], [], marker="o", linestyle="None", markersize=8,
           markerfacecolor=sim_colors[i], markeredgecolor="none",
           label= (sim.name.split('-')[-1]) if i != 0 else "A-" + sim.name.split('-')[-1])
    for i, sim in enumerate(g4)
]

def make_velocity_comparison(xlim_v=(2,110), ylim_v=(2,110),
                             xlim_b=(2,130), ylim_b=(2e6,8e9),
                             xlim_e=(-0.1,1.1), ylim_e=(-0.1,1.1),
                             title=None):
    
    # --- Base Setup ---
    fig = plt.figure(figsize=(7.1*3, 5.5*4), constrained_layout=True)
    fig.set_constrained_layout_pads(wspace=0.0, w_pad=0.0)
    sf_main, sf_leg = fig.subfigures(1, 2, width_ratios=[14, 1])
    axes = sf_main.subplots(4, 3, gridspec_kw=dict(wspace=0.04, hspace=0.02))

    # --- 1. NFW Profiles Load & Setup ---
    nfwx_m, nfwy_m = np.loadtxt("C:\\Home\\Astro\\TFG\\Cruz25\\nfwmean.txt",delimiter=',', unpack=True)
    nfwx_u, nfwy_u = np.loadtxt("C:\\Home\\Astro\\TFG\\Cruz25\\nfwup.txt",delimiter=',', unpack=True)
    nfwx_d, nfwy_d = np.loadtxt("C:\\Home\\Astro\\TFG\\Cruz25\\nfwdown.txt",delimiter=',', unpack=True)
    
    # NFW line generation for Vmax vs Vfid
    p, _ = np.polyfit(nfwx_m, nfwy_m, 1, cov=True)
    m_nfw_v, b_nfw_v = p
    p, _ = np.polyfit(nfwx_u, nfwy_u, 1, cov=True)
    m_nfw_u_v, b_nfw_u_v = p
    p, _ = np.polyfit(nfwx_d, nfwy_d, 1, cov=True)
    m_nfw_d_v, b_nfw_d_v = p
    x_v = np.linspace(0, 300, 300)
    y_v      = m_nfw_v * x_v + b_nfw_v
    y_up_v   = m_nfw_u_v * x_v + b_nfw_u_v
    y_down_v = m_nfw_d_v * x_v + b_nfw_d_v

    # NFW line generation for Eta_rot vs Eta_bar
    x_e = np.linspace(-10, 10, 300)
    y_e      = m_nfw_v*np.ones_like(x_e)
    y_up_e   = m_nfw_u_v*np.ones_like(x_e)
    y_down_e = m_nfw_d_v*np.ones_like(x_e)

    # --- 2. Observations Definition & Plotting ---
    lt_usable_v =[g for g in lt_galaxies if not np.isnan(g.eta_rot) and g.v_max < 500]
    sparc_usable_v =[g for g in sparc_galaxies if not np.isnan(g.eta_rot) and g.inclination > 30 and g.v_max < 500]
    
    lt_usable_b =[g for g in lt_galaxies if not np.isnan(g.eta_rot) and g.v_max < 500 and not np.isnan(g.m_bar) and g.e_m_bar > 0 and g.e_v_max > 0]
    sparc_usable_b =[g for g in sparc_galaxies if not np.isnan(g.eta_rot) and g.inclination > 30 and g.v_max < 500 and not np.isnan(g.m_bar) and g.e_m_bar > 0 and g.e_v_max > 0]
    
    lt_usable_e =[g for g in lt_galaxies if not np.isnan(g.eta_rot) and g.v_max < 500 and not np.isnan(g.eta_bar)]
    sparc_usable_e =[g for g in sparc_galaxies if not np.isnan(g.eta_rot) and g.inclination > 30 and g.v_max < 500 and not np.isnan(g.eta_bar)]

    alpha_obs = 0.3
    fit_used = p1
    i_lelli, s_lelli = fit_used.coef
    v_range = np.logspace(np.log10(xlim_b[0]), np.log10(xlim_b[1]), 100)
    m_lelli = 10**(s_lelli * np.log10(v_range) + i_lelli)

    for k in range(4):
        ax_v, ax_b, ax_e = axes[k]
        
        # NFW
        ax_v.fill_between(x_v, y_down_v, y_up_v, color="grey", alpha=0.25, label="NFW", zorder = -1)
        ax_v.plot(x_v, y_v, color="grey", lw=2, zorder = -1, linestyle=":")
        ax_e.fill_between(x_e, y_down_e, y_up_e, color="grey", alpha=0.25, label="NFW", zorder = -1)
        ax_e.plot(x_e, y_e, color="grey", lw=2, zorder = -1, linestyle=":")
        
        # Vmax vs Vfid
        ax_v.errorbar([g.v_max for g in lt_usable_v],[g.v_fid for g in lt_usable_v], xerr=[g.e_v_max for g in lt_usable_v], yerr=[g.e_v_fid for g in lt_usable_v], marker = "D", label="LITTLE THINGS", color="#6b6ecf", linestyle="None", elinewidth=1, alpha = alpha_obs, markeredgewidth=0, zorder = 0)
        ax_v.errorbar([g.v_max for g in sparc_usable_v],[g.v_fid for g in sparc_usable_v], xerr=[g.e_v_max for g in sparc_usable_v], yerr=[g.e_v_fid for g in sparc_usable_v], marker = "h", label="SPARC", color="#a252a2", linestyle="None", elinewidth=1, alpha = alpha_obs, markeredgewidth=0, zorder = 0)

        # BTFR Plot
        ax_b.errorbar([g.v_max for g in lt_usable_b],[g.m_bar for g in lt_usable_b], xerr=[g.e_v_max for g in lt_usable_b], yerr=[g.e_m_bar for g in lt_usable_b], marker = "D", label="LITTLE THINGS", color="#6b6ecf", linestyle="None", elinewidth=1, alpha = alpha_obs, markeredgewidth=0, zorder = 0)
        ax_b.errorbar([g.v_max for g in sparc_usable_b],[g.m_bar for g in sparc_usable_b], xerr=[g.e_v_max for g in sparc_usable_b], yerr=[g.e_m_bar for g in sparc_usable_b], marker = "h", label="SPARC", color="#a252a2", linestyle="None", elinewidth=1, alpha = alpha_obs, markeredgewidth=0, zorder = 0)
        ax_b.plot(v_range, m_lelli, color='k', linestyle=':', alpha=1, lw=1.5, zorder=2, label='Obs Fit')

        # Eta_rot vs Eta_bar
        ax_e.errorbar([g.eta_bar for g in lt_usable_e],[g.eta_rot for g in lt_usable_e], xerr=None, yerr=[g.e_eta_rot for g in lt_usable_e], marker = "D", label="LITTLE THINGS", color="#6b6ecf", linestyle="None", elinewidth=1, alpha = alpha_obs, markeredgewidth=0, zorder = 0)
        ax_e.errorbar([g.eta_bar for g in sparc_usable_e],[g.eta_rot for g in sparc_usable_e], xerr=None, yerr=[g.e_eta_rot for g in sparc_usable_e], marker = "h", label="SPARC", color="#a252a2", linestyle="None", elinewidth=1, alpha = alpha_obs, markeredgewidth=0, zorder = 0)

    # --- Compute Simulation Points & BTFR Dispersion ---
    def get_mass(s, radius):
        mgas = s.d.mass_in_sphere(force_recompute=False, center = s.v.center, radius = radius, particle="PartType0", auto_save=True)
        mstar = s.d.mass_in_sphere(force_recompute=False, center = s.v.center, radius = radius, particle="PartType4", auto_save=True)
        return mgas + mstar

    true_vmax =[]
    true_mass = []
    
    mk =["o", "s", "^", "D"]
    labels =[r"$V_{\phi}$", "W/ Pressure", "W/ Non-cte Disp.", "W/ Anisotropy"]
    
    for i in range(len(g4)):
        s = g4[i].snap[0]
        # Load your stored profile lists
        vm_list =[vmax_phi_list[i], vmax_corr_p_list[i], vmax_corr_d_list[i], vmax_corr_a_list[i]]
        vf_list =[vfid_phi_list[i], vfid_corr_p_list[i], vfid_corr_d_list[i], vfid_corr_a_list[i]]
        
        # Enclosed mass at R01_cgas holds constant for a given galaxy across all corrections
        r01 = s.v.r01_cgas.to_value()
        mbar = get_mass(s, r01)
        
        true_vmax.append(vm_list)
        # Store unit-stripped value natively if available to preserve np.log10 behaviour
        m_val = mbar.to_value() if hasattr(mbar, 'to_value') else mbar
        true_mass.append([m_val]*4)

        for k in range(4):
            ax_v, ax_b, ax_e = axes[k]
            
            ax_v.scatter(vm_list[k], vf_list[k], s=50, marker = mk[k], color=sim_colors[i], zorder = 3)
            ax_b.scatter(vm_list[k], mbar, s=50, marker = mk[k], color=sim_colors[i], zorder = 3)
            
            # Formulate Eta analytically with Vmax from our arrays
            eta_rot_val = vf_list[k] / vm_list[k]
            
            r_fiducial = vm_list[k] / 35.0
            vbfid_obj = s.d.v_fid(force_recompute=False, center = s.v.center, radius = r_fiducial, auto_save=True, particle="baryon")
            vbfid_val = vbfid_obj.to_value() if hasattr(vbfid_obj, 'to_value') else vbfid_obj
            eta_bar_val = (vbfid_val / vf_list[k])**2
            
            ax_e.scatter(eta_bar_val, eta_rot_val, s=50, marker = mk[k], color=sim_colors[i], zorder = 3)

    true_vmax = np.array(true_vmax)
    true_mass = np.array(true_mass)
    sig_obs = np.sqrt(np.mean((fit_used.yy - s_lelli * fit_used.xx - i_lelli)**2))
    
    # Calculate log-space residuals and print text on each row
    subscripts =[r"{V_{\phi}}", r"{\rm P}", r"{\rm D}", r"{\rm A}"]
    for k in range(4):
        ax_b = axes[k, 1]
        log_m_pred = s_lelli * np.log10(true_vmax[:-1, k]) + i_lelli
        residuals = np.log10(true_mass[:-1, k]) - log_m_pred
        bias = np.mean(residuals)
        sig = np.sqrt(np.mean((residuals - bias)**2))
        
        text_str = r"$\begin{aligned}"
        if k == 0:
            text_str += r"\sigma_{\rm obs} &= " + f"{sig_obs:.2f}" + r"\ \mathrm{dex} \\"
            
        text_str += r"\sigma_" + subscripts[k] + r" &= " + f"{sig:.2f}" + r"\ \mathrm{dex} \\"
        text_str += r"\langle r \rangle_" + subscripts[k] + r" &= " + f"{bias:.2f}" + r"\ \mathrm{dex}"
        text_str += r"\end{aligned}$"
        
        ax_b.text(0.05, 0.95, text_str, transform=ax_b.transAxes, fontsize=17, va='top', ha='left',
                  bbox=dict(facecolor='white', alpha=0.9, edgecolor='none'))

    # --- 4. Subplot Configurations & Legends ---
    gal_handles = [
        Line2D([],[], marker='D', linestyle='None', markersize=8, markerfacecolor='#6b6ecf', markeredgecolor='none', label='LITTLE THINGS'),
        Line2D([],[], marker='h', linestyle='None', markersize=8, markerfacecolor='#a252a2', markeredgecolor='none', label='SPARC'),
        Patch(facecolor='grey', edgecolor='none', alpha=0.7, label='NFW'),
        Line2D([],[], marker='None', linestyle=':', markersize=8, color='k', markeredgecolor='none', label='Obs. Fit')
    ]
    axes[0, 0].legend(handles=gal_handles, fontsize=18, loc="upper left", handletextpad=0.25, handlelength = 1, labelcolor="mfc")

    xy_label_fs = 20
    for k in range(4):
        ax_v, ax_b, ax_e = axes[k]
        ax_v.set_xscale("log")
        ax_v.set_yscale("log")
        ax_v.set_ylim(ylim_v)
        ax_v.set_xlim(xlim_v)
        ax_v.set_ylabel(r"$V_{\rm fid}$ [km/s]", fontsize=xy_label_fs)
        
        ax_b.set_xscale("log")
        ax_b.set_yscale("log")
        ax_b.set_ylim(ylim_b)
        ax_b.set_xlim(xlim_b)
        ax_b.set_ylabel(r"Baryonic Mass within $R_{\rm 0.1, HI}$[$M_{\odot}$]", fontsize=xy_label_fs)

        ax_e.set_ylim(ylim_e)
        ax_e.set_xlim(xlim_e)
        ax_e.set_ylabel(r"$\eta_{\rm rot} = V_{\rm fid}/V_{\rm max}$", fontsize=xy_label_fs)

        if k == 3:
            ax_v.set_xlabel(r"$V_{\rm max}$ [km/s]", fontsize=xy_label_fs) 
            ax_b.set_xlabel(r"$V_{\rm max}$ [km/s]", fontsize=xy_label_fs) 
            ax_e.set_xlabel(r"$\eta_{\rm bar} = (V_{\rm bar}/V_{\rm fid})^2$", fontsize=xy_label_fs) 

        # Method specific shape legend on the 3rd ax of each row
        sim_point_handles = [
            Line2D([], [], marker=mk[k], linestyle='None', markersize=8, markerfacecolor='none', markeredgecolor="k", markeredgewidth=1, label=labels[k])
        ]
        ax_e.legend(handles=sim_point_handles, fontsize=18, loc="lower right", handletextpad=0.25, handlelength = 1)

    # Side subfigure: Global overarching simulated color scheme
    sf_leg.legend(handles=sim_handles, fontsize=19, loc="upper right", frameon=False, fancybox=True, framealpha=1, edgecolor="#3B3B3B", handletextpad=0.2, labelspacing=0.92, borderpad=0.4, borderaxespad=0, handlelength = 1)

    if title:
        fig.suptitle(title, fontsize=16)

make_velocity_comparison()

In [ ]:
import numpy as np
import yt
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, SymLogNorm

def gen_sel_grid_fn(resolution_pc=50, radius_fn=None, z_fn=lambda s: 1):
    def sel_grid(s, altsim=None):
        rad_to_use = radius_fn(s, altsim=altsim)
        height = z_fn(s)
        v_bulk, normal = get_bulk_face(s, altsim=altsim, radius_fn=radius_fn)
        s.ds.kernel_name = "wendland4"

        center_kpc = s.v.center.to("kpc")
        width_arr = s.ds.arr([rad_to_use, rad_to_use, height], "kpc")
        left_edge = center_kpc - width_arr
        right_edge = center_kpc + width_arr

        res_kpc = resolution_pc / 1000.0
        dims =[
            max(1, int(np.ceil(2 * rad_to_use / res_kpc))),
            max(1, int(np.ceil(2 * rad_to_use / res_kpc))),
            max(1, int(np.ceil(2 * height / res_kpc))),
        ]

        agrid = s.ds.arbitrary_grid(left_edge, right_edge, dims=dims)
        agrid.set_field_parameter("center", center_kpc)
        agrid.set_field_parameter("normal", normal)
        agrid.set_field_parameter("bulk_velocity", v_bulk)
        return agrid
    return sel_grid

def get_or_build_euler_mesh(s, agrid_selector, cache_path=".", overwrite_cache = False):
    full_path = Path(cache_path) / f"{s.sim.name}_euler_mesh_cache.npz"
    if os.path.exists(full_path) and not overwrite_cache:
        print(f"Loading instantly from {full_path}...")
        with np.load(full_path, allow_pickle=True) as npz:
            data = {
                ("gas", "density"): (npz["density"], "g/cm**3"),
                ("gas", "velocity_x"): (npz["velocity_x"], "cm/s"),
                ("gas", "velocity_y"): (npz["velocity_y"], "cm/s"),
                ("gas", "velocity_z"): (npz["velocity_z"], "cm/s"),
                ("gas", "pressure"): (npz["pressure"], "dyne/cm**2"),
                ("gas", "HI_mass"): (npz["HI_mass"], "Msun"),
            }
            dims = npz["dims"]
            bbox = npz["bbox"]
            center = npz["center"]
            normal = npz["normal"]
    
    # =========================================================
    # INTERPOLATE SPH AND BUILD CACHE
    else:
        print("Cache not found. Depositing SPH to Arbitrary Grid (this can take at least 2 mins)...")
        agrid = agrid_selector(s)
        
        v_bulk = agrid.get_field_parameter("bulk_velocity").to("cm/s").v
        center = agrid.get_field_parameter("center").to("cm").v
        normal = agrid.get_field_parameter("normal")

        P_unyt = agrid["PartType0", "Pressure"]
        if getattr(P_unyt.units, "is_dimensionless", True) or str(P_unyt.units) == "1":
            P_unyt = agrid.ds.arr(P_unyt.v, "code_pressure")

        vx_rel = agrid["gas", "velocity_x"].to("cm/s").v - v_bulk[0]
        vy_rel = agrid["gas", "velocity_y"].to("cm/s").v - v_bulk[1]
        vz_rel = agrid["gas", "velocity_z"].to("cm/s").v - v_bulk[2]
        
        # Grab HI Mass explicitly
        hi_mass = agrid["PartType0", "HI_mass"].to("Msun").v

        density = agrid["gas", "density"].to("g/cm**3").v
        pressure = P_unyt.to("dyne/cm**2").v
        dims = agrid.ActiveDimensions
        bbox = np.array([[agrid.left_edge[0].to("cm").v, agrid.right_edge[0].to("cm").v],
            [agrid.left_edge[1].to("cm").v, agrid.right_edge[1].to("cm").v],
            [agrid.left_edge[2].to("cm").v, agrid.right_edge[2].to("cm").v],
        ])

        print(f"Saving arrays to {full_path}...")
        np.savez_compressed(
            full_path,
            density=density, velocity_x=vx_rel, velocity_y=vy_rel, velocity_z=vz_rel,
            pressure=pressure, HI_mass=hi_mass,
            dims=dims, bbox=bbox, center=center, normal=normal
        )
        
        data = {
            ("gas", "density"): (density, "g/cm**3"),
            ("gas", "velocity_x"): (vx_rel, "cm/s"),
            ("gas", "velocity_y"): (vy_rel, "cm/s"),
            ("gas", "velocity_z"): (vz_rel, "cm/s"),
            ("gas", "pressure"): (pressure, "dyne/cm**2"),
            ("gas", "HI_mass"): (hi_mass, "Msun")
        }

    # =========================================================
    # BUILD UNIFORM MESH
    ds_mesh = yt.load_uniform_grid(
        data, dims, bbox=bbox,
        length_unit="cm", mass_unit="g", time_unit="s"
    )
    ds_mesh.force_periodicity()
    
    # Store geometric parameters cleanly inside the dataset so fields can access them!
    ds_mesh.parameters["euler_center"] = ds_mesh.arr(center, "cm")
    ds_mesh.parameters["euler_normal"] = normal
    
    return ds_mesh

def add_euler_fields(ds_mesh):
    # Add gradients natively on the uniform mesh
    ds_mesh.add_gradient_fields(("gas", "pressure"))
    ds_mesh.add_gradient_fields(("gas", "velocity_x"))
    ds_mesh.add_gradient_fields(("gas", "velocity_y"))
    ds_mesh.add_gradient_fields(("gas", "velocity_z"))

    def _hi_density(field, data):
        # Convert HI mass to density by dividing by cell volume
        hi_mass = data["gas", "HI_mass"].to("g").v
        cell_vol = data["index", "cell_volume"].to("cm**3").v
        hi_density = hi_mass / cell_vol
        return data.ds.arr(hi_density, "g/cm**3")
    
    ds_mesh.add_field(("gas", "HI_density"), function=_hi_density,
                      sampling_type="cell", units="g/cm**3", force_override=True)

    def _euler_accel_radial(field, data):
        center = data.get_field_parameter("center").to("cm").v
        normal = data.get_field_parameter("normal")

        x = data["index", "x"].to("cm").v - center[0]
        y = data["index", "y"].to("cm").v - center[1]
        z = data["index", "z"].to("cm").v - center[2]

        r_dot_n = x * normal[0] + y * normal[1] + z * normal[2]
        r_cyl_x = x - r_dot_n * normal[0]
        r_cyl_y = y - r_dot_n * normal[1]
        r_cyl_z = z - r_dot_n * normal[2]

        r_cyl_mag = np.sqrt(r_cyl_x**2 + r_cyl_y**2 + r_cyl_z**2)
        r_cyl_mag[r_cyl_mag == 0] = 1e-30
        
        # Safe density division to prevent edge explosion
        rho = data["gas", "density"]
        safe_rho = data.ds.arr(np.maximum(rho.v, 1e-30), rho.units)

        # Acceleration = (v dot nabla) v + (nabla P) / rho
        ax = (data["gas", "velocity_x"] * data["gas", "velocity_x_gradient_x"] +
              data["gas", "velocity_y"] * data["gas", "velocity_x_gradient_y"] +
              data["gas", "velocity_z"] * data["gas", "velocity_x_gradient_z"] +
              data["gas", "pressure_gradient_x"] / safe_rho)

        ay = (data["gas", "velocity_x"] * data["gas", "velocity_y_gradient_x"] +
              data["gas", "velocity_y"] * data["gas", "velocity_y_gradient_y"] +
              data["gas", "velocity_z"] * data["gas", "velocity_y_gradient_z"] +
              data["gas", "pressure_gradient_y"] / safe_rho)

        az = (data["gas", "velocity_x"] * data["gas", "velocity_z_gradient_x"] +
              data["gas", "velocity_y"] * data["gas", "velocity_z_gradient_y"] +
              data["gas", "velocity_z"] * data["gas", "velocity_z_gradient_z"] +
              data["gas", "pressure_gradient_z"] / safe_rho)

        # Project radial component
        a_r = (ax * r_cyl_x + ay * r_cyl_y + az * r_cyl_z) / r_cyl_mag
        return a_r

    ds_mesh.add_field(("gas", "euler_accel_radial"), function=_euler_accel_radial,
                      sampling_type="cell", units="cm/s**2", force_override=True)

    def _vc_euler_model(field, data):
        # Calculate r_cyl manually to avoid unit/alias mismatch issues
        center = data.get_field_parameter("center").to("cm").v
        normal = data.get_field_parameter("normal")

        x = data["index", "x"].to("cm").v - center[0]
        y = data["index", "y"].to("cm").v - center[1]
        z = data["index", "z"].to("cm").v - center[2]

        r_dot_n = x * normal[0] + y * normal[1] + z * normal[2]
        r_cyl_x = x - r_dot_n * normal[0]
        r_cyl_y = y - r_dot_n * normal[1]
        r_cyl_z = z - r_dot_n * normal[2]

        r = data.ds.arr(np.sqrt(r_cyl_x**2 + r_cyl_y**2 + r_cyl_z**2), "cm")
        a_r = data["gas", "euler_accel_radial"]
        
        return np.sqrt(r * np.abs(a_r))

    ds_mesh.add_field(("gas", "vc_euler_model"), function=_vc_euler_model,
                      sampling_type="cell", units="cm/s", force_override=True)

    def _euler_accel_inward(field, data):
        return -data["gas", "euler_accel_radial"]

    ds_mesh.add_field(("gas", "euler_accel_inward"), function=_euler_accel_inward,
                    sampling_type="cell", units="cm/s**2", force_override=True)

def run_plot(sim, idx, grid_selector_fn):
    s = sim.snap[idx]
    
    # 1. Build and process mesh
    print(f"Generating arbitrary grid & uniform mesh for {sim.name}...")
    #agrid = grid_selector_fn(s)
    ds_mesh = get_or_build_euler_mesh(s, grid_selector_fn, cache_path=r"C:\Home\Astro\TFG\Figures\Circ_models\Mesh_cache")
    add_euler_fields(ds_mesh)

    # 2. Extract Data
    cg = ds_mesh.covering_grid(0, ds_mesh.domain_left_edge, ds_mesh.domain_dimensions, num_ghost_zones=1)
    cg.set_field_parameter("center", ds_mesh.parameters["euler_center"])
    cg.set_field_parameter("normal", ds_mesh.parameters["euler_normal"])

    # ====================================================================
    # 3. MASS CONSERVATION CHECK (Restored)
    # ====================================================================
    cell_vol = ds_mesh.index.get_smallest_dx()**3
    grid_mass = (ds_mesh.all_data()["gas", "density"].sum() * cell_vol).in_units("Msun")
    
    #rad_kpc = (ds_mesh.domain_right_edge[0] - ds_mesh.domain_left_edge[0]) / 2.0
    sp = s.ds.region(s.v.center, ds_mesh.domain_left_edge, ds_mesh.domain_right_edge)
    particle_mass = sp["PartType0", "Masses"].sum().in_units("Msun")
    
    print("\n--- MASS CONSERVATION CHECK ---")
    print(f"Total SPH Particle Mass in region: {particle_mass:.3e}")
    print(f"Total Grid-Smoothed Gas Mass:      {grid_mass:.3e}")
    print(f"Difference: {abs(grid_mass - particle_mass)/particle_mass * 100:.2f}%\n")

    # ====================================================================
    # 4. HARD METRICS PRINTING (Restored)
    # ====================================================================
    mid_z = cg.shape[2] // 2
    
    rho_slice = cg["gas", "density"][:, :, mid_z].v
    P_slice = cg["gas", "pressure"][:, :, mid_z].v
    ar_slice = cg["gas", "euler_accel_radial"][:, :, mid_z].to("km**2/s**2/kpc").v
    vc_slice = cg["gas", "vc_euler_model"][:, :, mid_z].to("km/s").v

    print("--- 2D SLICE METRICS (Midplane) ---")
    print(f"Density[g/cm^3]      : Min={np.min(rho_slice):.2e}, Max={np.max(rho_slice):.2e}, Median={np.median(rho_slice):.2e}")
    print(f"Pressure[dyne/cm^2]   : Min={np.min(P_slice):.2e}, Max={np.max(P_slice):.2e}, Median={np.median(P_slice):.2e}")
    print(f"Accel_R[(km/s)^2/kpc]: Min={np.min(ar_slice):.2e}, Max={np.max(ar_slice):.2e}, Median={np.median(ar_slice):.2e}")
    print(f"Vc (Euler)[km/s]        : Min={np.min(vc_slice):.2e}, Max={np.max(vc_slice):.2e}, Median={np.median(vc_slice):.2e}\n")

    # 5. Plotting
    fig, axes = plt.subplots(2, 2, figsize=(12, 10), layout="constrained")
    extent = [
        cg.left_edge[0].to("kpc").v, cg.right_edge[0].to("kpc").v,
        cg.left_edge[1].to("kpc").v, cg.right_edge[1].to("kpc").v
    ]

    im0 = axes[0,0].imshow(rho_slice.T, origin="lower", extent=extent, norm=LogNorm(), cmap="viridis")
    axes[0,0].set_title("Grid Density")
    fig.colorbar(im0, ax=axes[0,0], label="g / cm$^3$")

    im1 = axes[0,1].imshow(P_slice.T, origin="lower", extent=extent, norm=LogNorm(), cmap="magma")
    axes[0,1].set_title("Grid Pressure")
    fig.colorbar(im1, ax=axes[0,1], label="dyne / cm$^2$")

    im2 = axes[1,0].imshow(ar_slice.T, origin="lower", extent=extent, norm=SymLogNorm(linthresh=10, base=10), cmap="RdBu_r")
    axes[1,0].set_title("Eulerian Radial Acceleration ($a_r$)")
    fig.colorbar(im2, ax=axes[1,0], label="(km/s)$^2$ / kpc")

    # Filter out wild extremes for the colorbar maximum so the disk structure is visible
    vmax_vc = np.nanpercentile(vc_slice, 95)
    im3 = axes[1,1].imshow(vc_slice.T, origin="lower", extent=extent, vmin=0, vmax=vmax_vc, cmap="plasma")
    axes[1,1].set_title("Reconstructed $V_c$ (Eq. 9)")
    fig.colorbar(im3, ax=axes[1,1], label="km / s")

    for ax in axes.flat:
        ax.set_xlabel("X (kpc)")
        ax.set_ylabel("Y (kpc)")
    plt.show()

# Example usage:
agrid_selector = gen_sel_grid_fn(resolution_pc=50, radius_fn=r01_rad, z_fn=lambda s: 1)
#run_plot(g4[4], 0, agrid_selector)

In [ ]:

class MeshSnapshotProxy:
    """A lightweight proxy to make ds_mesh compatible with astroflow's `snapshot` signature."""
    def __init__(self, ds):
        self.sim = [ds]
        self.idx = 0

def run_astroflow_projection(ds_mesh):
    # Wrap the uniform grid dataset for astroflow
    mesh_snap = MeshSnapshotProxy(ds_mesh)
    
    # ---------------------------------------------------------
    # 1. HI Mass Projection (Column Density) - Unweighted
    # ---------------------------------------------------------
    data_args_hi = af.settings.DataConfig(
        axis=np.cross(ds_mesh.parameters["euler_normal"],[1,0,0]),
        center=ds_mesh.domain_center,
        width=(ds_mesh.domain_width[0].to('kpc').to_value(), "kpc"),
        unit="Msun/pc**2",
        x_unit="kpc",
        y_unit="kpc",
        up_vector = ds_mesh.parameters["euler_normal"],
        density=True,
    )
    style_args_hi = af.settings.StyleConfig(cmap="viridis", norm = "log", vmin = 5e-2, vmax = 5e2)
    io_args = af.settings.IOConfig(show=True, save=False)
    
    print("Plotting HI Mass Projection via astroflow...")
    af.plot.proj(
        mesh_snap, 
        ("gas", "HI_density"), 
        data_args=data_args_hi, 
        style_args=style_args_hi, 
        io_args=io_args
    )
    
    # ---------------------------------------------------------
    # 2. Kinematics Projections (Weighted by HI Mass!)
    # ---------------------------------------------------------
    data_args_kinematics = af.settings.DataConfig(
        axis=np.cross(ds_mesh.parameters["euler_normal"],[1,0,0]),
        center=ds_mesh.domain_center,
        width=(ds_mesh.domain_width[0].to('kpc').to_value(), "kpc"),
        weight_field=("gas", "HI_mass"),  # <--- Forces yt to weight the LOS by cold gas
        unit="km/s**2",
        x_unit="kpc",
        y_unit="kpc",
        up_vector = ds_mesh.parameters["euler_normal"]
    )
    
    print("Plotting HI-Weighted Radial Acceleration...")
    # Add symlog parameters if supported by your style config, else fallback to standard
    style_args_ar = af.settings.StyleConfig(cmap="RdBu_r", force_symmetry=True) 
    af.plot.proj(
        mesh_snap, 
        ("gas", "euler_accel_radial"), 
        data_args=data_args_kinematics, 
        style_args=style_args_ar, 
        io_args=io_args
    )

    print("Plotting HI-Weighted Euler Vc...")
    data_args_kinematics.unit = "km/s"
    style_args_vc = af.settings.StyleConfig(cmap="plasma", vmin=0) 
    af.plot.proj(
        mesh_snap, 
        ("gas", "vc_euler_model"), 
        data_args=data_args_kinematics, 
        style_args=style_args_vc, 
        io_args=io_args
    )

# =================================================================
# Execution
# =================================================================
# 1. Load the mesh from the cache builder we made
cache_filename = f"euler_mesh_{g4[4].name}.npz"
agrid_selector = gen_sel_grid_fn(resolution_pc=50, radius_fn=r01_rad, z_fn=lambda s: 1)
ds_mesh = get_or_build_euler_mesh(g4[4].snap[0], agrid_selector, cache_path=r"C:\Home\Astro\TFG\Figures\Circ_models\Mesh_cache", overwrite_cache=True)
add_euler_fields(ds_mesh)

# 2. Plot with astroflow
run_astroflow_projection(ds_mesh)

In [ ]:
def make_euler_circ_fn(color="#2730d6", label="Full Euler Eq.", log_x=False, cache_path=r"C:\Home\Astro\TFG\Figures\Circ_models\Mesh_cache", overwrite_cache=False):
    def euler_circ(snap, agrid, radii, nbins, adapt):
        # 1. Build the uniform mesh and attach Eulerian fields
        ds_mesh = get_or_build_euler_mesh(snap, agrid_selector, cache_path, overwrite_cache=overwrite_cache)
        add_euler_fields(ds_mesh)
        
        # 2. Extract all data from the uniform mesh
        ad = ds_mesh.all_data()
        
        # 3. Transfer the geometric parameters
        ad.set_field_parameter("center", agrid.get_field_parameter("center"))
        ad.set_field_parameter("normal", agrid.get_field_parameter("normal"))
        
        # The bulk velocity is already subtracted during `build_mesh_from_agrid`.
        # We set it to 0 here so yt doesn't try to subtract it a second time
        ad.set_field_parameter("bulk_velocity", ds_mesh.arr([0.0, 0.0, 0.0], "cm/s"))
        
        # 4. Profile the uniform mesh using astroflow
        args = af.settings.DataConfig(
            n_bins=nbins, 
            x_unit="kpc", 
            unit="cm/s**2", 
            bin_extrema=[(radii[0], radii[1])], 
            log=log_x, 
            accumulate=False, 
            weight_field=("gas", "HI_mass") # Weight the cells smoothly by mass
        )
        
        #profile = af.data.profile(
        #    ad, 
        #    ("index", "cylindrical_radius"), 
        #    ("gas", "vc_euler_model"), 
        #    data_args=args
        #)

        profile = af.data.profile(
            ad,
            ("index", "cylindrical_radius"),
            ("gas", "euler_accel_inward"),
            data_args=args
        )
        r = profile.x.to("cm")
        ain = profile[("gas", "euler_accel_inward")].to("cm/s**2")
        vc = np.sqrt(np.abs((r * ain).to("km**2/s**2").v))
        
        return {
            "label": label, 
            "x": profile.x.in_units("kpc").v, 
            "y": vc,#profile[("gas", "vc_euler_model")].in_units("km/s").v,
            "style": {"color": color, "linestyle": "-", "linewidth": 2.5, "zorder": 10} 
        }
        
    return euler_circ

In [ ]:
# Initialize your Euler profiler
euler_circ = make_euler_circ_fn(label="Euler $V_c$", overwrite_cache=False)

# Create your selectors
disk = gen_sel_disk_fn(z_fn=lambda s: min(s.v.h_e_HI.to_value(),s.v.r01_cgas.to_value()), radius_fn=r01_rad, set_vbulk=True)
sphere = gen_sel_sphere_fn(set_vbulk = True, set_normal = True, radius_fn = r01_rad)
agrid_selector = gen_sel_grid_fn(resolution_pc=50, radius_fn=r01_rad, z_fn=lambda s: min(s.v.h_e_HI.to_value(),s.v.r01_cgas.to_value()))

# Combine your functions into lists
circ_fns =[
    all_circ,
    v_phi_fn,
    euler_circ
]

# Map the appropriate selector to each function
sel_fns = [sphere,disk,agrid_selector,disk]

# Run the pipeline!
for sim in g4:
    plot_velocity_curve(
        sim, 
        r01_rad, 
        sel_fn=sel_fns,
        circ_fn=circ_fns, 
        ax_2_fn=[n_cil_gas], 
        nbins=100, 
        adaptive_bins=True, 
        path_prefix=r"C:\Home\Astro\TFG\Figures\Circ_models\Circ_Euler_test", 
        save_fig=True, 
        save_data=True, 
        ylim=[-10, 60], 
        legend_kwargs={"frameon": True, "labelcolor": "mfc", "loc": "upper right", "fontsize": 13, "ncol": 3, "framealpha": 0.9}
    )